# Test domain samples — VOV và Nhà thuốc Long Châu, 20 URL/domain

Chạy các ô từ trên xuống (**Run All**). Kaggle cần bật **Internet**; chỉ dùng CPU.
Notebook nhúng sẵn **40 URL**: 20 URL của `vov.vn` và 20 URL của
`nhathuoclongchau.com.vn`. Không chọn `vov2.vov.vn` hay `tiemchunglongchau.com.vn`.
Mẫu reservoir ngẫu nhiên không trùng URL, seed 42, giữ nguyên ID/URL từ
[`links_corpus_vi_en_zh.parquet`](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/blob/71be0e1239a19dfd7a717e2bc13e3bec21bd0334/links_corpus_vi_en_zh.parquet) (1,220,598 dòng).
Nguồn có 9,784 URL VOV và 79,598 URL Nhà thuốc Long Châu.
Không cần clone repo, tải Parquet hay upload code Python.

Dùng snapshot crawler và probe hiện tại: tải trực tiếp URL mẫu, retry,
giới hạn tốc độ, tải HTML/PDF, trích xuất và kiểm tra chất lượng (QC).
Hai domain chạy song song; từng domain xử lý tuần tự 20 URL.
Kết quả sẽ khác theo thời điểm và môi trường mạng.
HTTP User-Agent là Mozilla/Chrome; Accept và Accept-Language được đặt theo trình duyệt.
User-Agent không đảm bảo hết 403; kết quả thực tế được giữ nguyên.

- **VAO_DUOC**: lấy được HTTP 2xx và không gặp challenge; xem thêm trạng thái trích xuất/QC.
- **KHONG_VAO_DUOC**: đã chạy nhưng không lấy được trang đích dùng được.
- **CHUA_CHAY**: chưa có checkpoint, không tính là lỗi.

`robots_http_status` là HTTP của robots.txt; `http_status` là HTTP của trang.
Chế độ probe tải trực tiếp trang để kiểm tra truy cập, không dùng robots để chặn bước này.
Requests và Selenium được ghi riêng; HTTP 200 vẫn cần kiểm tra nội dung.
20 mẫu mỗi domain chưa đại diện cho toàn bộ URL trên website.


Phạm vi cố định: **`vov.vn`: 20 URL; `nhathuoclongchau.com.vn`: 20 URL**.

`NETWORK_CHECK_DOMAINS` chỉ gồm hai hostname này. Mặc định `MAX_NEW = None`
để chạy đủ 40 URL, log tiến độ mỗi 5 URL.
`BROWSER_TEST_MODE = 'failed_only'` thử Selenium cho URL Requests lỗi,
trang JS, text rỗng hoặc QC lỗi. Có thể đặt `off` để chỉ thử Requests.


## 1. Cấu hình

Mặc định chạy đủ **40 URL (20/domain)** đã nhúng, với `WORKERS = 2`
và `MAX_NEW = None`. Nếu giới hạn `MAX_NEW`, lần chạy đó chưa kiểm tra đủ mẫu.
Checkpoint đã có, kể cả lỗi, được bỏ qua khi chạy lại ô crawl.
Muốn thử lại toàn bộ URL, chạy lại ô cấu hình để tạo thư mục mới.
`RESUME_RUN_DIR` chỉ tiếp tục run có cùng danh sách 40 URL và snapshot code;
run 97 hostname cũ cần tạo run mới (`RESUME_RUN_DIR = None`).
`RECOVERY_RUN_DIR` có thể trỏ tới run cũ đã giải nén để tái sử dụng raw hợp lệ
cho ID/URL trùng mẫu, đồng thời tải các URL còn lại trong run mới.


In [ ]:
TARGET_DOMAINS = ('vov.vn', 'nhathuoclongchau.com.vn')
SAMPLES_PER_DOMAIN = 20
SAMPLE_SEED = 42

USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'

WORKERS = 2
DELAY = 1.0
LOG_EVERY = 5            # Tiến độ mỗi 5 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
NETWORK_CHECK_DOMAINS = TARGET_DOMAINS
BROWSER_TEST_MODE = 'failed_only'  # off / first_each_domain_and_failures / all
PUBMED_SELENIUM_FIRST = True  # PubMed chạy Selenium trước, bỏ Requests bài viết
BROWSER_USER_AGENT = None  # None: dùng User-Agent thực của Chrome đang chạy
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None       # None: Selenium Manager tìm hoặc tải Chrome phù hợp
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
PAGE_ATTEMPTS = 3
MAX_NEW = None           # None: toàn bộ; 10: thêm tối đa 10 URL
RESUME_RUN_DIR = None    # Hoặc đường dẫn thư mục run đã có
RECOVERY_RUN_DIR = None  # Run cũ: tạo run mới, reuse raw 2xx và thử lại các URL lỗi

from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import uuid

if sys.version_info < (3, 11):
    raise RuntimeError('Cần Python 3.11 trở lên.')
if type(WORKERS) is not int or WORKERS < 1:
    raise ValueError('WORKERS phải là số nguyên dương.')
if type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('ATTEMPTS phải là số nguyên dương.')
if isinstance(DELAY, bool) or not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if RECOVERY_RUN_DIR and RESUME_RUN_DIR:
    raise ValueError('Recovery tạo run mới; không đặt đồng thời RESUME_RUN_DIR.')
OUTPUT_BASE = (Path('/kaggle/working') if sys.platform != 'win32' and Path('/kaggle/working').is_dir()
               else Path('/content') if sys.platform != 'win32' and Path('/content').is_dir()
               else Path.cwd() / 'runs')
RUN_DIR = (Path(RESUME_RUN_DIR).expanduser().resolve() if RESUME_RUN_DIR else
           OUTPUT_BASE / ('domain_probe_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
                          + '_' + uuid.uuid4().hex[:6]))
if RESUME_RUN_DIR and not (RUN_DIR / 'manifest.json').is_file():
    raise ValueError('RESUME_RUN_DIR phải chứa manifest.json của notebook này.')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', RUN_DIR)

DIRECT_PUBLIC_PROBE = True


## 2. Cài dependencies

Cài các package crawler cần trong môi trường hiện tại. Phiên bản thực tế
và log pip được lưu trong output. Khi resume, dùng phiên bản đã lưu.


In [ ]:
import subprocess
import importlib.metadata

packages = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld', 'ipython']
if DIRECT_PUBLIC_PROBE and BROWSER_TEST_MODE != 'off':
    packages.append('selenium')
requirement_file = RUN_DIR / 'notebook_requirements.txt'
install_args = ['-r', str(requirement_file)] if requirement_file.exists() else packages
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    encoding='utf-8', errors='replace',
)
(RUN_DIR / 'pip_install.log').write_text(installation.stdout or '', encoding='utf-8')
if installation.returncode:
    print(installation.stdout)
    raise RuntimeError('Cài dependencies thất bại; xem pip_install.log.')
versions = {name: importlib.metadata.version(name) for name in packages}
if not requirement_file.exists():
    requirement_file.write_text(''.join(f'{p}=={v}\n' for p, v in versions.items()), encoding='utf-8')
print(json.dumps(versions, indent=2))


## 3. Danh sách mẫu và snapshot crawler

40 mẫu giữ nguyên ID/URL nguồn: 20 VOV và 20 Nhà thuốc Long Châu.
Mỗi domain có `sample_index` từ 0 đến 19. Snapshot được kiểm tra SHA-256
trước khi import; notebook kiểm tra số mẫu từng domain trước khi crawl.
Manifest ghi file nguồn, revision, hash, seed và phạm vi mẫu để đối chiếu lại.


In [ ]:
SAMPLES = [{'id': 704563,
  'url': 'https://vov.vn/suc-khoe/4-loai-rau-ban-nen-tranh-trong-mua-mua-post973112.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 706114,
  'url': 'https://vov.vn/suc-khoe/bo-gan-tam-binh-to-chuc-gay-quy-tang-benh-nhan-gan-ngay-viem-gan-the-gioi-post1035616.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 1},
 {'id': 706377,
  'url': 'https://vov.vn/suc-khoe/cach-cho-con-bu-khi-me-duong-tinh-voi-covid-19-879521.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 2},
 {'id': 706678,
  'url': 'https://vov.vn/suc-khoe/cay-thuoc-quy-khong-trong-cung-moc-tua-tua-khap-vuon-it-nguoi-biet-tan-dung-post1244137.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 3},
 {'id': 706928,
  'url': 'https://vov.vn/suc-khoe/co-nen-an-bi-lon-thuong-xuyen-post1102200.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 4},
 {'id': 707097,
  'url': 'https://vov.vn/suc-khoe/cuu-song-be-trai-nguoi-nhat-bi-chan-thuong-so-nao-khi-dang-trong-thoi-gian-cach-ly-880653.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 5},
 {'id': 707299,
  'url': 'https://vov.vn/suc-khoe/dau-mua-khi-lay-lan-the-nao-post946723.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 6},
 {'id': 707479,
  'url': 'https://vov.vn/suc-khoe/dieu-gi-xay-ra-khi-ban-ngung-uong-nuoc-ngot-post1010466.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 7},
 {'id': 708399,
  'url': 'https://vov.vn/suc-khoe/loai-cay-kho-ngot-nhu-duong-la-vi-thuoc-quy-viet-nam-co-nhieu-lai-it-duoc-chu-y-post1263715.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 8},
 {'id': 708669,
  'url': 'https://vov.vn/suc-khoe/loai-qua-sieu-giau-tinh-bot-an-vao-lam-cham-lao-hoa-lai-ngua-du-benh-post1213223.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 9},
 {'id': 708775,
  'url': 'https://vov.vn/suc-khoe/loai-rau-viet-ra-cho-la-thay-la-than-duoc-bo-xuong-ngua-ung-thu-post1129703.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 10},
 {'id': 709340,
  'url': 'https://vov.vn/suc-khoe/nam-dau-bep-suy-da-tang-sau-3-ngay-che-bien-thit-lon-post1036522.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 11},
 {'id': 709402,
  'url': 'https://vov.vn/suc-khoe/nau-nuong-thuc-pham-khong-dung-cach-lam-tang-muc-cholesterol-khi-ban-su-dung-post978352.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 12},
 {'id': 709448,
  'url': 'https://vov.vn/suc-khoe/neu-ban-bi-nhip-tim-nhanh-hay-thu-nhung-loai-do-uong-nay-post972732.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 13},
 {'id': 709673,
  'url': 'https://vov.vn/suc-khoe/nguoi-viet-chi-tien-ty-moi-nam-mua-tao-do-an-kieu-nay-thanh-dau-doc-co-the-post1155173.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 14},
 {'id': 709963,
  'url': 'https://vov.vn/suc-khoe/nhung-lam-tuong-ve-vaccine-covid-19-khien-nhieu-nguoi-khong-tiem-chung-875456.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 15},
 {'id': 710876,
  'url': 'https://vov.vn/suc-khoe/su-dung-hieu-qua-thuoc-nham-trung-dich-egfr-tki-the-he-3-cho-benh-ung-thu-phoi-post1012057.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 16},
 {'id': 711104,
  'url': 'https://vov.vn/suc-khoe/tam-dem-bang-nuoc-lanh-lien-tuc-tai-trai-nguoi-dan-ong-mat-thinh-luc-hoan-toan-post1206514.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 17},
 {'id': 712207,
  'url': 'https://vov.vn/suc-khoe/viet-nam-co-nu-hoang-cua-cac-loai-trai-cay-moi-ngay-mot-qua-bo-ngang-to-yen-post1191038.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 18},
 {'id': 713042,
  'url': 'https://vov.vn/xa-hoi/lan-dau-tien-cac-tram-y-te-tai-tphcm-co-300-loai-thuoc-post1126708.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 19},
 {'id': 213425,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/bi-di-ung-thoi-tiet-nen-boi-thuoc-gi-nhanh-khoi-53300.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 214251,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/bi-trat-khop-lung-phai-lam-sao-cach-nhan-biet-trat-khop-lung.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 1},
 {'id': 217630,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/cac-trieu-chung-khi-mac-lao-phoi-la-gi-va-benh-lao-phoi-co-nguy-hiem-khong-55060.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 2},
 {'id': 219040,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/cach-dung-chi-nha-khoa-co-can-chuan-nhat-59725.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 3},
 {'id': 219815,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/cach-lam-toi-den-bang-noi-com-dien-don-gian-68993.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 4},
 {'id': 220312,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/cach-phan-biet-tri-va-ung-thu-hau-mon.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 5},
 {'id': 225132,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/chup-mach-mau-so-hoa-xoa-nen-dsa-la-gi-dung-trong-chan-doan-benh-nao.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 6},
 {'id': 229499,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/dau-tuc-tinh-hoan-nhung-khong-sung-co-nguy-hiem-khong-67806.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 7},
 {'id': 230885,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/dieu-tri-viem-va-cap-tinh-o-tre-va-bien-chung-neu-khong-chua-tri-kip-thoi.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 8},
 {'id': 238312,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/hop-thuoc-chong-soc-gom-nhung-gi.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 9},
 {'id': 246897,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/moi-mo-xong-nen-an-gi-kieng-an-gi-dam-bao-suc-khoe-70824.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 10},
 {'id': 249470,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/nghien-ruou-nguyen-nhan-trieu-chung-va-cach-xu-tri-51141.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 11},
 {'id': 250706,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/nguoi-tre-van-co-nguy-co-cao-bi-gai-dot-song-co-42025.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 12},
 {'id': 254687,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/nhung-hau-qua-cua-benh-suy-dinh-duong-o-tre-duoi-5-tuoi-41829.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 13},
 {'id': 264708,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/tac-hai-cua-thuoc-nhuom-toc-la-gi-nhung-dieu-ban-can-luu-y-de-nhuom-toc-an-toan-66186.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 14},
 {'id': 265237,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/tai-sao-nhan-vien-van-phong-thuong-gap-van-de-dau-co-vai-gay.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 15},
 {'id': 266033,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/tap-yoga-nen-tap-luc-nao-la-tot-nhat-57374.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 16},
 {'id': 268911,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/thuoc-kalimate-la-thuoc-gi-nhung-dieu-can-biet-khi-su-dung-thuoc-kalimate-69001.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 17},
 {'id': 281452,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/xet-nghiem-hpv-o-nam-gioi-va-nhung-thong-tin-can-biet-56801.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 18},
 {'id': 282916,
  'url': 'https://nhathuoclongchau.com.vn/benh/mnemophobia.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 19}]

SOURCE_PROFILE = {'rows': 1220598,
 'id_min': 583,
 'id_max': 7466161,
 'duplicate_ids': 0,
 'hosts': 94,
 'host_counts': {'article.iiyi.com': 729,
                 'ask.39.net': 32772,
                 'ask.familydoctor.com.cn': 1,
                 'baby.39.net': 3368,
                 'bachmai.gov.vn': 2783,
                 'baidianfeng.familydoctor.com.cn': 60,
                 'baoangiang.com.vn': 10738,
                 'baocantho.com.vn': 12016,
                 'baochinhphu.vn': 2302,
                 'baodanang.vn': 14500,
                 'baogialai.com.vn': 3290,
                 'baohaiphong.vn': 16286,
                 'baolangson.vn': 1981,
                 'baonghean.vn': 15941,
                 'baophutho.vn': 1288,
                 'baoquangninh.vn': 2912,
                 'baoquangtri.vn': 5483,
                 'baotayninh.vn': 1198,
                 'baothanhhoa.vn': 782,
                 'baovinhlong.com.vn': 5113,
                 'benhviennhitrunguong.gov.vn': 1200,
                 'benhvienvietduc.org': 425,
                 'bingli.iiyi.com': 5,
                 'byby.39.net': 37,
                 'cancer.39.net': 736,
                 'care.39.net': 1282,
                 'dantri.com.vn': 2063,
                 'ek.39.net': 448,
                 'fitness.39.net': 185,
                 'fk.39.net': 928,
                 'food.39.net': 594,
                 'gan.39.net': 294,
                 'gc.39.net': 59,
                 'giadinhonline.vn': 16729,
                 'gk.39.net': 90,
                 'hanoimoi.vn': 1283,
                 'health.people.com.cn': 50,
                 'heart.39.net': 1055,
                 'hellobacsi.com': 11747,
                 'jb39.com': 527,
                 'jbk.39.net': 2033,
                 'khoahocphothong.vn': 8143,
                 'laodong.vn': 31282,
                 'man.39.net': 382,
                 'medlatec.vn': 24762,
                 'mega.vietnamplus.vn': 2,
                 'nhandan.vn': 5225,
                 'nhathuoclongchau.com.vn': 79598,
                 'nk.39.net': 762,
                 'pf.39.net': 85,
                 'phunusuckhoe.giadinhonline.vn': 27577,
                 'phuyen.baodaklak.vn': 6252,
                 'pmc-ecm-healthblog.beta.pharmacity.io': 5333,
                 'pubmed.ncbi.nlm.nih.gov': 240000,
                 'qihuangzhishu.com': 29,
                 'qy.familydoctor.com.cn': 2,
                 'shen.39.net': 325,
                 'sj.39.net': 197,
                 'special.vietnamplus.vn': 1,
                 'suckhoecongdongonline.vn': 154503,
                 'suckhoedoisong.vn': 85823,
                 'suckhoeviet.org.vn': 5627,
                 'tamanhhospital.vn': 4261,
                 'test.pmphai.com': 2989,
                 'thanhnien.vn': 72111,
                 'thaythuocvietnam.vn': 2927,
                 'tiemchunglongchau.com.vn': 13281,
                 'tienphong.vn': 3021,
                 'tnb.39.net': 717,
                 'tuoitre.vn': 3893,
                 'vietnamnet.vn': 15543,
                 'vnexpress.net': 2055,
                 'vov.vn': 9784,
                 'vov2.vov.vn': 6899,
                 'wei.39.net': 225,
                 'woman.39.net': 687,
                 'www.120ask.com': 98160,
                 'www.a-hospital.com': 4701,
                 'www.baidu.com': 11,
                 'www.cnkang.com': 40046,
                 'www.familydoctor.cn': 2,
                 'www.familydoctor.com.cn': 26819,
                 'www.msdmanuals.cn': 905,
                 'www.pharmacity.vn': 69,
                 'www.qdnd.vn': 3700,
                 'www.sggp.org.vn': 1853,
                 'www.vietnamplus.vn': 3695,
                 'www.vinmec.com': 26129,
                 'www.wujue.com': 708,
                 'www.youlai.cn': 4139,
                 'www.zydcd.com': 204,
                 'youmed.vn': 7189,
                 'zhongyibaodian.net': 6061,
                 'zysjonline.com': 6591},
 'first_1000_host_counts': {'bachmai.gov.vn': 999, 'special.vietnamplus.vn': 1},
 'sample_host_counts': {'article.iiyi.com': 0,
                        'ask.39.net': 0,
                        'ask.familydoctor.com.cn': 0,
                        'baby.39.net': 0,
                        'bachmai.gov.vn': 0,
                        'baidianfeng.familydoctor.com.cn': 0,
                        'baoangiang.com.vn': 0,
                        'baocantho.com.vn': 0,
                        'baochinhphu.vn': 0,
                        'baodanang.vn': 0,
                        'baogialai.com.vn': 0,
                        'baohaiphong.vn': 0,
                        'baolangson.vn': 0,
                        'baonghean.vn': 0,
                        'baophutho.vn': 0,
                        'baoquangninh.vn': 0,
                        'baoquangtri.vn': 0,
                        'baotayninh.vn': 0,
                        'baothanhhoa.vn': 0,
                        'baovinhlong.com.vn': 0,
                        'benhviennhitrunguong.gov.vn': 0,
                        'benhvienvietduc.org': 0,
                        'bingli.iiyi.com': 0,
                        'byby.39.net': 0,
                        'cancer.39.net': 0,
                        'care.39.net': 0,
                        'dantri.com.vn': 0,
                        'ek.39.net': 0,
                        'fitness.39.net': 0,
                        'fk.39.net': 0,
                        'food.39.net': 0,
                        'gan.39.net': 0,
                        'gc.39.net': 0,
                        'giadinhonline.vn': 0,
                        'gk.39.net': 0,
                        'hanoimoi.vn': 0,
                        'health.people.com.cn': 0,
                        'heart.39.net': 0,
                        'hellobacsi.com': 0,
                        'jb39.com': 0,
                        'jbk.39.net': 0,
                        'khoahocphothong.vn': 0,
                        'laodong.vn': 0,
                        'man.39.net': 0,
                        'medlatec.vn': 0,
                        'mega.vietnamplus.vn': 0,
                        'nhandan.vn': 0,
                        'nhathuoclongchau.com.vn': 20,
                        'nk.39.net': 0,
                        'pf.39.net': 0,
                        'phunusuckhoe.giadinhonline.vn': 0,
                        'phuyen.baodaklak.vn': 0,
                        'pmc-ecm-healthblog.beta.pharmacity.io': 0,
                        'pubmed.ncbi.nlm.nih.gov': 0,
                        'qihuangzhishu.com': 0,
                        'qy.familydoctor.com.cn': 0,
                        'shen.39.net': 0,
                        'sj.39.net': 0,
                        'special.vietnamplus.vn': 0,
                        'suckhoecongdongonline.vn': 0,
                        'suckhoedoisong.vn': 0,
                        'suckhoeviet.org.vn': 0,
                        'tamanhhospital.vn': 0,
                        'test.pmphai.com': 0,
                        'thanhnien.vn': 0,
                        'thaythuocvietnam.vn': 0,
                        'tiemchunglongchau.com.vn': 0,
                        'tienphong.vn': 0,
                        'tnb.39.net': 0,
                        'tuoitre.vn': 0,
                        'vietnamnet.vn': 0,
                        'vnexpress.net': 0,
                        'vov.vn': 20,
                        'vov2.vov.vn': 0,
                        'wei.39.net': 0,
                        'woman.39.net': 0,
                        'www.120ask.com': 0,
                        'www.a-hospital.com': 0,
                        'www.baidu.com': 0,
                        'www.cnkang.com': 0,
                        'www.familydoctor.cn': 0,
                        'www.familydoctor.com.cn': 0,
                        'www.msdmanuals.cn': 0,
                        'www.pharmacity.vn': 0,
                        'www.qdnd.vn': 0,
                        'www.sggp.org.vn': 0,
                        'www.vietnamplus.vn': 0,
                        'www.vinmec.com': 0,
                        'www.wujue.com': 0,
                        'www.youlai.cn': 0,
                        'www.zydcd.com': 0,
                        'youmed.vn': 0,
                        'zhongyibaodian.net': 0,
                        'zysjonline.com': 0}}

BASELINE_DOMAINS = {'vov.vn': {'error_type': 'not_tested',
            'requested_additional': 0,
            'selected_additional': 0,
            'shortfall': 0},
 'nhathuoclongchau.com.vn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0}}

EXPECTED_URL_COUNT = 40
EXPECTED_DOMAIN_COUNT = 2
BASELINE_ARCHIVE = ''
BASELINE_ARCHIVE_SHA256 = ''
SOURCE_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
SOURCE_FILE = 'links_corpus_vi_en_zh.parquet'
SOURCE_REVISION = '71be0e1239a19dfd7a717e2bc13e3bec21bd0334'
SOURCE_URL = 'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/blob/71be0e1239a19dfd7a717e2bc13e3bec21bd0334/links_corpus_vi_en_zh.parquet'
CRAWLER_SHA256 = '7f2d93d72b8b251cba34dd8080b41e649d47549b0147fa358958822e99eda6ed'
CRAWLER_PAYLOAD = 'c-qZ<ZIj%_dEoc`70{}Dn4xB|vrAHumK@nMMcXV%ltNL?rIr%}JA>JQc4jaFF#F==esEk)9AA7DUtY?&ODa-)sif>UauQ!I7JcUrROx@P%9P|!_ZRMY>qa-wn4P8NyD0Gz19U(A_Vn}KPp_}9e>M4HlD>BPt~bb{qsc}(o1A#_WRezvcPI7US;X0iH;MM*N#60M>5%>{vM8A)vwbhkyks_vkKs93UteDvW$Dxl!_lHxWN{dJ$#kA(g%{0cX%Q7kI?LDA)Vpkd9%XsVbq48V5)Y_^s&q45%!)Ycc*A%UEhfb<85FAUY_Q0(cvb}5lCE=aoW;@bjWnIyI*td6B8BQv9uCs!d=eM&P}eWw<6@HRse1Be8qK2pIO9^osECVX8mltuHw+*Cel49z*W+oFOoBy`OmsJBU|2pZ(ooe?1K<BzGFQ*zC?7*N)KC5gUIf#)h@fJmzs#fI5C)p77jNh3Og&GdV$6M+!;>+rH}HczXVGk!PW3O{oqUX4*T2%iL0srxxM+Ei40N;kiGD0-u3_c%3uBOrSuy}*p&?Mu7TE;HMKhzi2k+ialUWBJ^7$kwtfE=Emljk$Qx)#wt1l&!_zilkacO=MWm$RzOARwnc^Tm1FN&Bx4yqpkKA?C(AKxh_2j1mHInpc|C6fp+O$Jj;hN?^ZaS@It!?m@q-n#qd?bq*kJ<mUU!C!mp20p!(el3|yqRp#8*K7S2Y?bsV_wL;Dwz@(0ng=hQzjDnxe*Q|^yD<k0{jGTKm896bdighk%g=kQSH5!ZwO2ddBsqw^FUNy}wC&v-L;LaO)@1;I|L46KjiM}(D*oEt*T4Asz3|4DZ$Tqv`~KS68tfu(Q4Ct`oiz{s%VHSLOe1wLOOIL#uK}3{?I6k1QJMj6wc1jBnC!!}v~W>vD1yd8K907pKF<Y$aeOQlRDGDlIK;9o#8UxrloVq^RDKzx^LW<svpv6!i!$Ouj<yJxG%)ITKDX_=+Kq~Inhe6dlOoQkm5yhO0bjsem0i$p5j&8`Ob>=h*5V&|?_Rb58UQpeFT(VIesYx}p3VUf)x=SSvuGL<$N-<b=e&_0oa${8AnGJTfZ1xF1;upkmtcaUECK4lJ*K{vpR#y9i3YLsl}E_MktWFa(!IB3c#gqixnQ`M&hr+OgR#tVB-c?sNRr-{qDc<pL$W?AdfV+Fp5abz`HNz-@q*u0O-|BiShi^612upJ@|Ll(IPMV1Y>Pm`6!fY<vEoE}3EMMo9)s@(NIP1_STfg0+A!_@{h99pE_=wW7;e+3#wp+p5FmRzEk&T{Q-FnNMFFG_7&l3e;!I=P3^&ar`C8n<u?ir31l%ylVo;Xz?|kW|-{CR>QZ2H7KZDM9Xb9F@_0EKz#X-K<Yi0iK{ldGy=ys!S|2Zt^3G}u(C|&yg?DTTyi~;J#cy^D2yvUMyt4-5)jMIm30-g-&K)bEV$oS{bOV}zcVDM`F(nX<m7`_eQH+Ll&c`f|7>G{BWlM(FN09O>oN*gR!<cj(oCBQqr8ci1QEg;@m%YS`Q0BQH438K~sk_IpG6rgJ`!UZxLn$*EKP2or6&Eg|(V*?ruWb*QKkqu(2fBZ@a!3g*CJHZ>wDUkcL%<nLS`6$Q-V~_(vW-5pqx%W)Z+x3$n&L?1v-^X5n7zFeoC{E^a4g@w#K<og4qg914O5(||g|*vMc<C(MI-Y~*6a!0QXkIb|iEHTLvVe>RjiY$An9g!F<Ybuhpm<A0dwNVARkTNa@3;9MHBzV;%odXgNK_!4;fUsgfS!RAgP<@Q6fG#z4nXLdp8(pxBus#w<GW=;crD3uktc@>z^wtWCABcOU)KGmmayw08@BlMD9L~pH++27d+YA2UbHC2DG<pMFUh@pF&ZU<1n3)~^*l>~nLzW`yhGTK$P&E~4D!U=i{U%`h>O7(DK@tTRrVrygS!D14k&LH(hDF?<?PX#8-vDbxD-%{3I|-Y)rA#KVX`I(a;&nv%s6690&@eR*bH_)K<MDYwiPzPE}kMd*vG#!E{b{n`JK(p-QU@yjb9Qn^S|2kkNn#+8o8Qsn6Vx26ly<{oypXz|3HQW*&j(+Hc@0?6Pmi*-reqNN@&*Lb=l>Leeb!Rw^bcvzL>VmMGOwHbFlf_-gS_by50I9%;I&}0FyT|kXP>|f?tCS6IhmkW|rZno}UO62@k6%oP5^1_0A%iY#>trN;F9ar0B3*lf}3#vSe>T?+^#DT+UxdK#GtXF<!m1038V?7WJF}#ms~FB!_L8!Pb`Ew>w^!TRj>l*bf*NRkJ2ckIF_qSvmNBs#3!=daI}MdUf51qO97U`p<@`mnVa*-V==^xJ&;C;f>>=Vp1UBAEvO%I3=Y9*us4xggcIBby~1&ceblDG5`UrqqZgJWvwM!vt#euE$d@tn_YaD;9#2f$Q(3UV(@BGR8gzcB$+12XsubVE;Ev21cxg}73!;opMz))g7~m?iu8X+PZg219goP6&n?hbd^od51eO8^Mv%DxhXUjj5di+T56s6HeY;*~B-I*wta`mm<@x~sBK>A;;;r5DWXH8KW)3*>C^hP-*SKGblr*0m4>oLX5>5Apk(cdwnQ%~sUPM`elsiQg0VMyZC^Eu%=ua{X={q3E9Z<gjlB+;ESKqbhM75Bk(Q9P3Eowo%Sw_;{1fd}r0gp?85^ZcG!wrUp`&@nkq*i=9m@M+-Fm7w%VdUv{kpp2%WkY#9VPT|6z5rGWTM{^AON67BdVYr?!obzb>po9byKil?vYF~?onE?>nM+H8e5Tm&S-DoKgUi>;*eu)>&29swb1~GHNgB%lS4b>9H6Wc4(MJ_h<hG3s;z6ex{8?adUBwqpd2Bmgw?fXDP@mZqg2fyKnU<oCFglr=zA+cvp|%pzSJiY`O%uVLfsoif!Bu0J%%w`TR23#H2wCLfN>Jxi_ua>-f#n0rtxRAba85&>G294KT3!%4;lYwM<#Ti_pQB^@bJ>O#F@ZE-V_m39Y#E+$oiwwhw<2V~H-0bxu@Hg+N@$lfrIU`DqDmQBCI)uaG+nk*WKGln_7)&d7!^p@(6EE(5A&7B_{P6+`^$@LaS!aY$D7K=i#rQUougzQOAOL%zR2+%aVUI6hF#wnA+HyF0x3qeFl_2PogBu%^g4P$#rm>i*my)qGJ)YVfE<l2!BVLW;yXzS@ej|BQMlrn(<IpJprV}sO$ZfrmZ+_wo1rFR^evrZzd_P~Us~#*S4<fM3j~p?{dbPlwH1unS%%am0>C^}C_kyNVYZkl+Fw#0(6bMdbdg({t1jO*>}K!^>G5vgl77%&s%d~t(9#LfZ`Hsi_5-j5HVZtD)=!}-6U#Fi>$*#60T>P)zElckpe-(9n~qW=&~j;80x0b&bS}mnK&q^JgCf29$SSw`O21ang#5c?GPwQnCMu27Bq!Y}@EiJoCOoiJMn@qUrO;vt%yYZj^{)4<5~!FVnijO(x+r>Gb}2Og$)0yL8r?*{R%b5-%SVsA3c)Z&V>h}9T2O<zFB{F;PN_=lqzHoeo7tRTdEYH;HO(B30Cz;=+_7{#fF6S#DT@Z&dPg@+o)|DEpw+|gm+wc(4CnxJ^ZLB;Tnozs@aGjlq5{SMDq1f)nne2{s%}sTgb1oxF9kFP78eG>n}3C+;u;sUDqBhd`D#D|nV%r-!A0%})bmEnR3JpXvWK)>A(=(Z!E%?S13(|*D?Q>Wb-;BtWl#s)_Gtf7E9dGL_C)Cy=4LL-x|%@3R^x}7!Y*dQq?`luW%$hR`G?TlIT*<b(~NxD#0Z*%1n8l(jdsm5cr^lUGnO@6TN_NGJoj#*dzJUft>ZyFM}vS6MzorSz`&9s3_%r|j7S~^Vu8NT0=$boFbP{CHkb>k01CkLP<~v@QMU{<JcfXzkxQ06`VyM8;;gX`FdFQ@Bs%GB1t22<UjqLF;Oga$cM0wDpgQECev9m>QzsTu4EFU5CN)GK69DdQT}6=ck<|qGg08J*IAFZaPQty#5OBWtV%GpNf?=ZZzbGaIqdH?W9D1TL$+POd;-a{OGU{@~HlWJDp{9OB&QSR(x)_Tym5-*XfdsGRg5OqBQNk8iXCEy4m773Ut4tSQu-ZeD_&m*%0;D5lKsIT%*&2(Eq+C~kA^tnM)p7EU@;~2=9JocDA6A8D;!_FCe7icN65UKl<=a&u8R%I8GG3lp!WpQh9ghqRqO(>O0qt7T++i;^Y{gU^tX5>~5p!|<a|;$7DOMU4ZCwl_3cx=xLV6%2=^OV5q?_(aX5~{D#+gPmhDI`y!nh_(lT`v#Yc*hkYf;^>ti`gv7_*NSDOd4&NxO~xMMM&7>D7G|A}Cswo`6yKAeXjhAosuqv3p_`;8(cI&N{~boMQtMLXw?O*;<Ry#OfElK_{J}_Sj|!xF*y-i}%q~eL`jt1Fy(ZU5>RvWt1zC>w+d#h^@9o7+jqeqxl@f3}AuOy3w3y4<JL58dMI36f~;hSFpY;6G#a%m2wP6@TRoWKBo@E4MamQOeuls)PD=q&5axTAfCa7ed`8v;Kl$MHz~@nikstNI_ZEML&gLEu{j@(uD#Ri1}}ClZC;|MtNvNL8o^L+n`h+<6mByB6>*#n^*kw;RLgeeaEK9Gm`m5RIgMO`{fe<L5RY{c)cMI-<hM(&B$YN@+wuxzrL6L54&uX6T0;gR0EV37C+s#MvE9`-il9+FBl*F50T}v~E0^0B0S%*p%1G9RxEm3w(cH5XTXW(Ssd8^21&R+buuTd15bK67*wg3H+H8Vgc!;h*kW%O13opb?qbM{IV4}?wF@<^_Mrb9oQQE4^WycyWRBgck>nqgjsYPK~;e~Fy!5ye#!GdHS%3}+=vwNk#+wEiIPPjiw_o7J?kv1oCG0US-tl;>aDkd0PR`yhwBKu&fK?Zd)$}3Tkm+=xu$)P;QFNUzY!1fdODpm~I3>J=;pO9PhRD>0~K9LZIIp{vKVsgR~wkJ+CyE0NYl#Uho)s#*XV}Neqr0SuP*zC!}T0t~;X94(P(2{r}i(-9OLq{8Aw;a2-L6Z_pQ>c+bC<qY5a}ga@We&&<u}%U7(fuy3gvV?O)W&fHOcv<DQ3QMyylaMq7<*=H2O!gSlUN02O8RQ@C^BB)Jdw@9(fKNNnp3Flc(~iBjhA}MW?~gwz1Uprov0}ofCvPkce9Eug1V4T;&|S|4pu%^=SjP9kgZla?@)4itCWI_dcytxLI<owZi|d#1{*9e7<WoRxYo2%51kdrYNhJ9MYzvs<{c)VhB*O4eB44qz6hgb2RFTKcCcw7xKiC=m(Y@mk}cYV!9+OJmaIUbV2U`Jiv0!s_ea|FldBLiaNwv3a8KG`yqF<TDZr#ZAyX@C8E456#UepzaVio}_EL-KtM&ZVtCyc|H?Iyi@*GFj5Uh4%SyU{k_)E)Nr_Jl00#`(j6guTb5KaowFvU2Ufj<kOoq46jkv1Hd8vW&+scamDvCzLOS7~oF(vVtJgPyc6I<7iCH+P#CR~f99O!zVixUBcTwvFhC^sN!CVU)_+BqDG}8I?7Da)A(I7dxeB4O?e1p!oaXCb*q$L&+KIbuTAmIF;H079%KB*e1`ZyElqj3sW*pHuMjoNw;wY#4v^u<(2IhJKojq<t5m`)WsO7(gMhCe;0MZjT<A_ojA?iB}n3A7fVK^ev0ZXI(9Ssm~4iIdcC~0bGx&&OV0BP(!~IuVUnlwmX-<V^ODXR189{a=y><Q<|U7>1EGPLIHI5J^ZSk}h#Qv;0cID4m%f8vLr}09Z3C-fP0h70*6c7ea7#>Fl7g!n8l7DNR>#%LB^#ieZf7S*AkDy%4Lj3T`@vws<^yHt&653OW(JGkSJbS~xj`Q>h<cMN1;ugUuUP^S>7k7xktT^fUHZ9315G(5)WpDEPXTcp<nNzyXU=v`HP(eQFoK==DxACB-rd=HzVy6+D*(dSDZ2O6=h~pX^Meiw(!+%})NIwOuKI#5UwL#}{d`3lp(Kg`z8FH;=`WE6G+Ydcf!`!wl%|_`3Z?{D<rp^s^qEyBWkCdrIYS}Vv?nkmC`ynl1y$N6IfH_%wmNRx@pIl*UD3Q4zf$w8j3+dJo`083S=7(Elqlm24<Kl(=bKm%E)ylqQKQsW_lN@5J6LmdZ@01_{nB>7*3^VOvDRSc>AdBaW3a-r4C{l8pO7B{)bZq0EI0^)nV9>)-`qjZFNQcE$F}VgAcMg9pTLf*<s)#)a)}6hz5>G$sPhsjngmT~pJ1QLCXv=koW|e@ej=RaenOzeoPw|BR8Z}qvxX+rs_1RgS5g)=?`9}_@jiuCo~whM%Lkjp9^NIUu|to3tGj*0yF~wV&gD_({-=}@hw0m}m}nulu|XMtY4$uvHd7*5v%G-|ptiFSztB^YW;*J7meh2Uzic3PN3qFouQ*j&wiR%_(}NxFko_Ux4MC(;4T<{$Db<0Br-koNyCf|Ycd$`Nr&S!cuPF9Lq{KknqVUu{TX9&3RT56oGb1kAOT_*o<&PBNzX`9vSi7m1A(M}i?O|Mwf2z3CSs0zl<~$3tQ`nAY!T#6$#`di9Rmxe4F%1}BK#?Unu7smCrii7zZu=bkF{2@f{bv-Hf)RP(06K0G8AVjoZgSMW;oZ(^$jZD2+k!eE4_xhb`yGJ^z(R6<_sHFf1;A5|X3j8dFW17{+I*AvSD6No+U6L4kb}BBgBXN9QSAi~I)_3`1P)`xmr_kN+{(7paWXwkv#gye9jb)ctX&k31&r~Ef^}rId$rYHMP0kx1qtPHcN_m)#y`J-e_r6IDvQFrsy62mMWJMma?H3S;sR^8gwtQ5{ktO7nnjJ~$oB?)-j?l9nO8!Suzrr}NBoV4PJsw!QDV+yG8>}HjFW=6ZWJw69)Tn}N5>2O#>Milqsz-z#{farhR|1io`KFTUVjE+stBtZN-f(fN**v|D%>gB*+Z>g#8^_BTs%?jp$i;Pz)MxM2I(~IhEoM7l0zpdXU|(&=vr2f&%3wEGn2^2^G3R-CjcwZ(DA{6Ullt?_X;@8wz}<|U*8aR9b2T{57@w08tpBO>pS8A$x{Y(=3i{<0L(<}lOjuh<=(wFJPoaVk^N~1{?@?2Vh`3NLh-AWz4m;!`~PydB{Hu*UN*6qgJaFIuG=)|H%`^&!n#m{a<(eBS6VTARN-%Szzcln7*p1k5b`GV*&Q`QKw4Uk7SpQk9ayT~Wa#blQK(f3xgppZv=&JqNa4*_8R+?p7@1+K5oWc!?G+_rB;96|vhvEibhO=#t^T<btb+4(am!6jmaQha?aQ@Ce|g;+-avZlJD|RwoW_LZjH$tBv33pwEe)3vs>{(|cGocH^m<M!&dv%ut>RGE^3XD;x)idPdoO}yBqKZ+zwKSR<h4}aHo%eu<|H}T3CRvHfe3n)gYB#BCa1Fi`xRZ!Pvv1Yu^Sf>?4z(gu`KRQu@G;n;j#>t8(zw+Zs77SWRJ3UmFfo$#-S_V>zZEB=)|rn3L2Yw`U%t`TQTW1DkWmN#pau}g+T;%s&JeX;yA5FK56@=bWJRPMuZ7TBwVbFFfqfP8Y7;>P>t;MbS^P9XK)rCe$3(m)Omx^swjyiMx@f2Y)6mS0edymxw5>6g5Zvsl*ZR;6E`f)?RLzk+ktr95eZ1!Mwz!ch;ue-yJ!oy*x<Kdn7a(9Uxv|?o&y>9np<Y9t+3Z~0d{d>F3u5J*5)LOdBbUx3Sm<d&_LJbI=0EJB2hG`cD&RhKz|iV)f~iW%(lZ1WCotPDT(SJ??EW<GB#l6FJD9;cAywadTNH<GM;gI!!RZZ!^J})ttKlLxS1sXONm15wa*5@3||l>5-1^*WQ%u2r`*C7!l>WlG&V+*G8Hr0Q;%y^L_a6^OT-pU$;tSk1oD`r&}!tYXl;XN7LqxzWuydiWdv!~tCF0;VMw>KNG&@dd%7vC8`4X&G1_Zg;{-bGQK_S>>-g5HWDbX_ASu8HRiz^7sJ`HS6(1>;mUSZ|rct@ZlBAAaA<c?-@{A})JgsI1npg&=^6i)tKU5A|hNZ{lg~|drT@-34Di*v%Zp9s)f|=-2<!Uy|JVqJ<E6CbLUOZ!U<{<O6=woMh6slYJD{j~}O0vkS%^+B2*!)Iu?FK^@uEwsSDSg)_Yc}83EA!)8)m7H$LtWxLy}0*mnWIc~byW{pqr>Z|oL%DTQXVfx%l6t5OSh>+_Up&fs?i`T=-ErH|H{7Z(5fz1wJHN6tIAZf7S&890Y4!I{vcHJ*w*JDc#)EW9uL-<Jbxm!lL{prXbOmGFhN(%Z)~N)n5QoO*{0|x{Bwo1XEp5WtBm?t@6{A!WcGIHa2XIrAqG$>N*GihL3&v5jL~^dc3LlH!75_f)9at+_%9%eL6G6|7mIqu5Gb<0gNme;LkhUlfni!Jq@=m5hl(cH{xvRrU8y8WvCL!Za1E7_uEe)KGq3Ki?#XqyWPw}llWR80iw#IcoQ&A+Tl#9-XEj~tGxsm)W8Y8bR-$<>lPe*Su=qof!cT4kUCm5LcVrk2PFzYMTxr!z3nctl;cTv0Q;F)JE#T`ku<2CNcus(2&s2&_$p8EbAhkMVL!Ks_**!Xnv4`?AWU7rh_Zg}L5Jn6(g=8>^Hz2XtGy>8wju#mv9Hwm2f=O_itQ4pS8l8^w=@`>6IV2QF=uFcV{oSU&m+9{n`g@iBK97G3@!=1Ycb_0)GM3msE5*M<utXMMza5$)-?QH2c#7E+r#S8$PN^N@`B@O0CS;}q1ij|Hj>!dMocCdZZA3-|q@5_F%eXn2=2~;QKWsk$sGRf9b@r}2*AeXt`%{(LwM1hW*9FCBcjW^}+F8(C6eknfV96*0r5K-Psqy)cb{(4X{N-IJ!OJSacBgKEF>NpNt95Z<)lMjLnd*taLBRAZtECII*X;QE1QRsR2#8&umz4_CNCmwO@U*<+EQ-}`S<*eX1(7Ap2(om{-UH8XH_qPq)3sKPNNq(k!K;yAR-&#Mh)#A3cFi$ZGQ;^|PaR>TXx|nbiHOTxeHVSD7_+8mjbN{zbMgcsd>u%wijo3C$XWV<@KelDggJL`=j%-^0B{>)Q^E0^LR~gDFYSII8TQr{<%cMdV!dDH9m4|;sxBog*{Y?7MvKIn9TTN$z}Yy}QlOhEz$Y7T&^W3ssTiqq8WNYf@5v-9HUO>Ru@0h?qNelIP@-O7#zP<*khb&A{-bwKybr(o;WuXc_w!3T@W0mm;ptXq`)t$r_sUtj_4#M+58GBQU+F3VKDVQ$YEaGp4TNJ(VcZ027ei+J&kZ*}H{JZ)Z$lDVLJK=5_&HJOX6rV($$Esyed;{qZaf7s(xYSU*_6&O5E5ZNI^YmT35r&^B)~KRCL!@N*=X>Y$f`m%N%=oN(TV(lOW|)573mNazqa?%h|b3j=>!txppy!;w!=6dz;=vgnB=H_h<QaqUgOXdp%k;)U$nR`$tCoG&+KB7Xx6;jrcHzQ8(<@Doej?Y5r&Y#B;LS;XZtXP8?w77Ktr0UQ(ht4YO50`_h|{tdKJSrT@wMV69~Q4ctdQCuk2#WApT&Q+!=JIweGEVN;0*P&oR067YH^O0O2eBC9v?^uL#uO<~e9=Eqcnvh#p{LyI+8_>!WI$<qTt5g%))|bgS7(OVa@Avbp_Lj%<+6DL)HM?!DW0yjO2O`t}{~H{ZH($G<@EB4Q;^3!)`WyKOn+ZraO3^EAvL9~u%}=!24H4gm9WqVSX|re#Ve&=eX0X;(o3-hS=C<TFxSCHYSWlxl~MUaHMbEX6cGD!x+gb1Av`oB!d4Cu>J`3S*B$50cCjFjw5<qIlY;gi$%b_r$Tcw!2Y&fMJfvxsQ&H0^^ObU&%;8x0E!nF33QbgQ~RN_iL6}UDsZgWgwpphxZ@>{sC64Y?^46A2Cb(%-WgdST%GsCY(u+uF9QU9Lk&yq{vFq&_M8bdu2FO0>`wgi<8uDQEnM|zp<{Mdd4`fYe)eU&ETQmw4&#aipa_mq=5zd%9e#R0_Bur!FEh(j?(lqq1jbkfFt*2mmxr-uAckU3(kAXGsWG)Syk)BZ-_o^N&JlOfF#M=!0-jhFnFtk??kxi!$VTSBrlihUI=<`rc!6Am#!{KJsFESH$#G-F}x);DI`_3h6$@_R46btu5Y_}fY^0T2bL1RUH@6x;ow;a^hU!G2ljS=<cM`LGngHK8QCN=#v6moz#x)lbODj$9`w&bm*ogH-lFeT4ZVaNPmd5K2kw!U()5zT(4}F&wh^Si$OrAKGU)abKKX)bKXeJ)O7Uh@F=9cb-hI&SpOQ97+OxXUpd@B)(M6S-_OP`uQs2@ke%0O%9NpFH>S}Pj^((a(HBXNs12{V>MT~=LtUhrX+YA&#R8aU;^`&Zj+(i>lAj|oZSN<4?`KSCg-je6Y#!ws(FWjn3$e|T50E+=Vm2Plx{h-l`_tbiV${()8drZZfrKVu-$<P=c-a|qX>)QwOh5&3lx^HO?bJiJbrfiaemjB6r`?n|G{Fjfv{kOmT`k(xtpZ!-{PkHs{fAGl<-^Ew$wkyuED#@-@!rOJDGQOH9<}xhRv7pL=@t@AGi;EiM*vZN?8(W=O>E~u{nM~orJ7R3gRavfjo((98aWpBc_xQWvNMBt`(Gr-kH2|Te`+i#(WZh5z_NXbI!EII@-lD5~-gxcy%NwNvaci~@yKwA9dpS)<M_ZcTSnRz91!mD<vd@{iGIZvkx41M)$vt3bn}N$;)TJt$t=+AS7a^|_{JGTL{Jh_Ba5$|{a3-x{++YS##$}ZSx<v_=%EElJ02;STX-|^xS*cJACOHBwiQsm>fUtV&r})p=dVli^O3gQxis1)9cL$It3(SHwaT7ghEm|Wk+tw&8zb%yJ5zVJ`Oj{4iA0&L)V~Af$n;MmSMlML(2V>2=W%|x*<|@*}op5;vTC+~CcIx8L=@PBLmNlN5nT|Qh#)dW>m{ZOjaZb4!U^(YPHkOi+C4;WcpxOva1j+pkRj6iXRh7>(E2@GP-Bpz}zmy)U56$3)Y2sdV_~5I%`7+VkaQt`>0x1F9ScDjMR^C<xHLQerg-+r!@?U=a_6{OPa%KXBnEV32@A@~Sqp;sqpZk_|{Z#V->BVwIjlfrxGFy#OWKWtilqoXM=Q=NeYjVKc!wkQkhS%rIuiO>+Ti#e#Iz>BqhqiD>zr+>Up#@WQt@0sw1K3BqL$UQ0$W2U6jo-+hucyRaZf!NqjmRWXNsB=BafTh=Xj$pwv*tcQy^h(Goxt4moSWPV@Q_a18IsSQ|G76_{?djBXq5z#EQJ}s>Idia9i~(8_K)n4WqHZhXi9D??jU9iC%3+jtx#AEJlImu9Hjx=StSzVE4r$ZHa1?j!VwxwO7Sb*fHNHRUGh}9OR!V=_5+e5go?A2dj9^|kMnjz%t2)z@f6U;fm)Ukfx%!f0fbzfgmaO8l$~JCsrh+mG%hNyHF=-UKg!`4*DA5gTOE{?g17*?JKNS_Nhzm}OM2&3*jZbe!x8MtbQ4Z!o$R&IQe9Z~21!3;&xJFflL!NOXlKQ6X`;UK7yQ<sO|hRG)A?gADG<8gpNaS7FK%Dn+UC^<Ji2+3!8NnswO^AE2LvS-QpQn6f|R@k?^X}~w|js6zTf|RYxn-qM*q3CU)?`Y^R~q}r238-X!#Va@WgICfBA(geSH9yt_vjx_&y+C?6=MK)m^2~kO0@ROBt>*(%m60t(h4HKdSm<0NUTfiNm(`_YUx%;|+W;R@Z0`%51&Rz1(G<j|c*a+I*o7@QOHzV}O|COmYF1XZnoBcbd~b*1)Q1QFSRoAcKf$%XY-!dvEc5rY}<s$>j8EnjS3Xt~^e)zUmWKiE;BPYmdZ6g4tj%31*XNFiSvWOb?A10<!p%`$FBN4FhQ&rnn_A0@>Csuo_*r$Jb@H%K2zl!k%3jtf+QO!WAE?b&b1h+fpwR?JRK?*VkFd77g{Sx#h~(GupF%wp|2SfZMS(#s$}b;%enQ&@}WY20En+S3~vz_k6j9fi62%w7pkn_0Iq3zMdgcXO4|2FCOOibZeNp7XuEyt2>*_qeXm?h;iGn1URq@j0Q*hQL<xm?<sajaix;k04$3wwOt&kEQDW{i*v@LI%GHvi6ys=A*Jb0@}}BK4I$8aP;OQm2@sz3;_PWR9Aq+@!3D;pzKvmv=w_tTiF-LlhE!E>&izgnVoVenQ_L47(Jmc+W&9JbkGiNV<h$SD<jrJ)C-5k|*G)0bcj0xsm2L0K2}`+`8yuah)03wCFC0&i0TU@*37x3au9WEWeKa(U^2WZ|iM}b)B9+Z_#?^yBPrW7-^u-Y$h)BWV>;l9)CU<S~Ox94w+FjVG<*`k5N6Rkw(oyPTmpT!2HdSGOArTZEj|x$du*E4jiG+0s7+Gd^HB8ZHWxZ%)z;qxo=YkE{Ht$}|=3Uyiok2McTQ~kEcbYa^SM?xKcrk=`BVx2|44mhq;_qM!F5hi9Qum$!M5u)?O+AV}$SHp=^qe$HnG-ANplsB~c6ybo+B>BYwx@8HweW3!8Aq#WvOS!A>;QmNhF0vK@s<0&45~|y?XfqciX7qyaUp1bg^sfkxTj6P3e9W-uzJhpUt<|vrrpnqq=?kQRaAtD>w2yoivn4<z>xvVdTm}NMxG08(sh*}6EE`^oas%u=dD)9EUm9wkQ$bqlZ(q8WFZd6IM-7~UE@Vo@sVE%?6E7CCv7lTOE_yS(zdA%CNQ8jWM<84(ROrjV@F#m;l0XD)A1?-6CDXyQ`e{EWyCg;4DGyN1@Uc4%?~=ae}8fLO1JA{LX`pjffp~1MqmV?I9`=PNEzNJJ<(Lc-7)-!RC|O1b%vB54bK5Wn7<rN28&0*5ifKFcz{P5=i^B3?k(ZhD`K7=97C=mx2Q(0YFOEP^i8p<EM?2wmg7gzZMdlufo;K=6e<_vHP$n-^y-H3gL;YZr~^dWx}vtfNg>(Z81JI+;D*VQYoNj>xk@!nap=ijM)7GA5NsBxofE4-D_j<+9fH$uj29fI!FGu;x+W~f*Tx~X#`-7a=O}e;fA_<0Jo=jtzx(J1kACvedmsJy!{2-KdmsMsqi;X@(~sW$=>5KmQgs79Of0}Ag)obObO{6kGHzJ8TOeP2GKK3iiJ0QFZg*+MbVZ}*JcE}dMGyKsHQJWKx+wwI%&+3bluAzXrp>{u@@@krolgQ3d7*0^F1P^~{G}OXY{MJpK(baBmMC!=mYOnQU6WXAxCsO<B2jnEE0-2Hgl3D%*@D`n8B^k(Icpr%$*P6?oI3@TNFX%@+uidg&x{39^|qWv4P&w@c%?eW<>caIdl+sgI|z;-!j&(042{mUp@5<b*OZkd6FLJUqV?Gje33S4QcpbyS(jYm*Rd;sDGuTW_Nz`XlRCnV?sm7D0Ej-AFLJzpx`a@wq_zf0EK;a-S5xb;kn8RhO|HjIdWyYi?2FP{ag+0r6PxV;)dM++Lb^rSgmXd%NYc|iI+;l4JkXMvPt$`~ZnN$1Rn2%Gig<wG;A%`MKg{4!rX~UpvNic1|LY$<`QaZu`RUJ}eD~W={_*?2_}RP2)}DOxUw`rN&p-b5pMCoNAAa)3zyHaf{u90ZqlcgT>0dwj%fJ8RgP%Y7=662*$xomBm%sn?{lCF)AAj#ZfBe0_{`5QVKlzidQ&hwM`tBz`{EH{=|LoIm{h%&M`oH+;kDq+!&!7Co-#_`s?-A(#{+%b^`qxkX`Fo%K<R?#l|C>+#@u&Fq!#~Co0^V%=_=6wK;k>}(hu`q_9{s}%?kifrD>}<`bNuK>@FIWo9xTS*;}5={dQ<pjEHbVm4_Pf1tzG#_=7aXm<)&rZOTn3t(Sp?RNHN{(UU21V69l|xg9~B~fF!jLnfRs6mh9SG1_&{+Ne2p>=6$3DO@P2p5YF)UW|$5#8}+Om;TBaJ!8Vnz_w#9(&=RI{9eB!eYNP#F=Xa(?<f$7R3buqqH_i$GlKm(Z&glaQ(3T!X-q4C>)?SGdzy+CC&zl)T1-7N`DsYLKshW8$m{f*ndx(eJL8N!%nCZQEEg679oQ{gycr`j4Fu!#?pMXaqdoaz@4LYYsPX#|-f9C!!w3Un=T&EW=J-7}gm6t^H5+_E}m{U5Yh;$cs{9+EfF;W=HY0l<&GC8Bm^!(pxeSQbhOg@0@jt@}Hfa_y9@8W~E(ZUyb(-b++1FSM5kC6;|_yA6hAE1pzT>}}<(6S5fF{L?@<P(@wIL{21p~FCb`QHEd<zM~HFaGU2Pk!`0Ag=QJ<G=jD$A9?0pM2}b4+_k~;2n$~{REl9<A;9&MC|bg?<Eh^t4IGZ1%B}OgTFoTsBGJ>&$Kl@QE!y&F4M*VZGblDgySS6#zK+kcY-~qBc0;~Tuj_BAReywc61mqgCk%vDGz@6gFpK8{jY!W?$;j-qB%qyMfTz~t{#s7{vWDLKw{Zfl7PM!xYP!<L6P(_$%!a>UnM4vEJ_a8xD~1T&YGQ1f_2EZ`MP8~Jffm*?&u5@70q{tA)|#BwyXBm&MH=KC8V-+x}(MYGH)#+pfHmiQyjwM!6v8KQ5>q$g2hx!K*)~78=#UM@2EuFBuGdCBgv%OZmS3Z&Wr;-A4s(dnY8(GiHzJRva<Th`_oI8sEjX(*6bT7s`pChIrR}8P)K95{?gp|mQPHwe@K@C&UOPm5Hm*o`fkkSOgFpttZU<~?-D{MiOVHpxvB{M>^|mx(tt486J#Yf4s@7l$aMyCg1TxN^2(*>`X2k9YSl<mtsabg<ouO7&=U*lNj=4qWLs8t6+7jLBl&XOVNX@e#34>tM$7E!vLjAZok}im&t=1Em>@LtG<-5`LFQ$ecU8&=){#<>y)!!DusH@6KoJ2bP<gB!v*n%Aw5U@wsBKL^qz^38Rx{|;>}96eN+v?loLbtWTFw}IY?2tGOeQMXv~<>Z>vU^#W?Cmyh{k|SX+#l2D_2(!vK`g)pq@b(8oW63B~NyHtqvJ)P8R1zPe{Vbi5qn`c?N9cW>aa|s42?JHd;R=5&*GY{WD6$wr&#wecvy>GKn1c0`_yEX_It6<m*ey6jS(_GIhu&A5(91wGc$pqCZw|JTC!S*PPd{I4=vHf4Mo|KuDwtI~V9H1%Z_(BfL$m$9E-}bEUpXk~-CW46P#SqRO!vN~PfDTeLr`n!EB%XU$(Wo;RteN7NFSBXHuO>b@jtOXuq7Al|jTQ<-nNgIO+QZkP9p%Nu0sr)$-F%8e`C?dyl?m$YwP4Z5ACOV=-V;bRSY<v#TnyCwwtigW~UrqaVOdlPPjQ7-({qMK4^{Ul==J?s1DJ>;t9M9Gp3QAjQ5;%uL8I85<gsIT6-`{wP}?;w9=Q-o&r<pPHxJ%?@bv_q4a$*XL?9%ygOXQgjG-mDx(<ptAg?z`i45i675rAvx;TNlGSfXq)o=bDA(IP;S{kYCDo$OryWp{G0_Tt@wL!31?yV&_ihJL1I!>vI|6Ck<doosOn!m4Xbmb`k-aA8}F^J61v>_}XV@W@x@_E|#izxV#Z{ggJ8a_!z?1a*41jcQD22<4#nIzii_XCxKJs+=$wH*SrOtJD9~snB>Z1hJ{%PfECIyWz0zc`3q)vsz>crgXXd<qq?BB@)5Zjj5rp$?uSo#(`Aw{bxz6>@TMczs)`O$QmEykU{{0)i=}S{TY%;%za8|#jIIM+hOWIX=Y2V&ShF&zsM>lpDp$5ghZI9mE>Bsdp27(Wb6dE6x?wU)@;4H4^hf&Dbf2)b%;hLGNdWUIo&5-WcSqO|pl>#5YnzCyko3s7`lMkF&g~N30wl&>vom#vsw}Hg4q*-iu{aO+UHS)Z!vvHVgQpCFRHTMxU__a~VhgsgBO_#9ql-N4?wAD`g(P)T$FudlI4|EQhu}>z%4R6OxE|MZ5c9n;;`JMlVd2(sJXpXMYhjz1ZV!`*Q77aF$1y`mLsMSRy;q3k_#w!_riMv>lrn~CHLKc~xf5itb^=dG*AgFh1U6;HpDm^_r7w+;_OhD*P8M+EFs^D|NQB-K00Bp*RW2=)soW(bR^o0SZ~EP;WDKxp8Y?hbR&;KT5K+#vC6L&-pyXJ9RD;O6Isx46JB|`sea4+=$mr?-spw55a8ujibg>+N-EBHzNc&7zXUB<-f)PJ0FQf#@C<{i&q0EhUQ!lZ5oI4lwZB6NfT13x>i|IUvL~oR+5*2nJPFgMS=}?6xXvU3{S`T00m|bZNuoj`iH#+cZ$9%5@x@N>_M70ZnQ<k5GsqNsrt^|;0(g4IS6c&`TaRw?AM9G76NzvXCMT)oYF7h#-XsF^eZF%o=ral9=rhAGQ;4eBeAhdKPG|+@1MAtn;1s$qffjf%C_?QZJxR}tn8MA>4;DS2Tz?9jq5)AlQqt;KeeAe$X$83V}4CF%8M#vQs+yRxBPtoDEhg$E#gk*eP@wfm*2?WcGR7()DDARtcG?>9@&DY<g{M?xF=@1mV91V-0CDA(+&N&Z3g4Z(8?Y9tV3M>SeI-Y5moOpDRABt&Y#)fC`)zTVNkAjkNNZbLWA5L>kw*mMiry_}y&f56Vb9;QM286|->AuCFBzgs@>LBA6`Bo|xBKb~&A{-B@UNkMAX?6{sA+Pf|K4^6<E0rm|lPd=j_*~wu#05#{<upGWWX1U@skt0Y07YX;pD=A}ihNwYzxqme=Zml2d<7YxD>t>I^=-`R%OcTTkO@GQ6rt#bn3)DFmV1<kmSSH3L#}~wj2Hu7Bp-Qf5}^R_DazpWOuTgwO6FoTm=&wsqKd^}IS@5@9y;>w)z@#n65jfcklD}pedmk+ZfBx4yptrJ442<5-GFu%LA{IN(_43(_~g8_;GCs->yAo(h$Zm2RMmRLu{4VCXcA?pMr92FRZ=+x<TEoa0hPiTaTOO|q?9l#v5r_#DUVskJ^sv;$^Tz=ucO@7U(rHD7p71p`>GhH=86>>Ro8igww75-Ft4F~Rw}3^e3C*8?UcW)te)Ow=g{q!H{o{ZWr}H5RZ4eUslvr=Lyi;-@ex;w^0e4GE8zlrVju+vsn%SezJ5ywF<QTwX4zs+>MR(GWS-8}YlC2Mo=g$!(dX`FD&n-}Ws^>sZn>4F6;l{7ZZ%CQBAM2v`IA1i0=Sp4wy3No#tPK*FDop!zh=aBH-j==9hM%&_388>X9OZME?s7V>1WD>l>|@Mvm}wsRGsYEV!*rADb<n5vUD;ZlO*EYGrOl1&*&IUqv-(dAtou5j-|HXhXjq2{fh!=xj{1nI7P+}G5E;a>YqCt=#V}Z7OEEM5&&*f;NyU+gCY&1U7Ha0jtg-iDyAqKfM3u#H6d=DvP129Y1p^(1)z|uyso2OIH~~ZfV~D{)M8!wB8c{kg=h}=dZx+|$Qp9ty^|1Q<Zar@aZHedF`X|a?8-w*%hM~I2b`8rkNx%-$1clhM!e_EsLm}gs`GPccR^3KZbaCJ)z)sch76+2uP)$s^ho@2D0nDc|Dc@U6E+cG@PSvVYNCc^`4SDx*rj}$ym=QJ)36j`8O_fCSD`Sg!Zs8^A|J{IXZG3zr4s8O#mWAdNf-9%c-NfHK3o$5Z%Ql#w1eMzTKlB{qL2-6z$$^7mR@nh5}|o$jVZ=?xwWCAMzl!ZK_=R!b&?!C#`;neJ|R@wwbIZqzE`moR(Xlx=Q6cWPO`9UA0Nx2X^Yo6_AYyCIN+M-h0T}4>{ysWi7^zBse_TQpYt(F3ZC099s9EiNGW<QP_CwS`AR5?fE<vj1`7+)s6vqN*NA>K+N4}<Sb<6P?lLSpdZ|+yW!o;)bs_+TgKBoiyiKjR5b}C8g$s1iR#@2q8c|@^nLDBeMQ!`g0tZkuJrLea>!1u{YQs==WH5Xzrv>dYbwa%Z#Jee)6V90rlV~R4WR$1^91`@dB~(I55=1dp13G{4D$fbl*`<pU2XQiK1utF&qDDt^1UByft^SHFfbs)46zW~4D9W|cA|)aLx(m@{h0enyr&}W^)RpH3s>;(@LlB3;RaQj<IhZfhPleLojRE{{F-R~Vkvj#4AY#a>0mq3TF!PwzE=W*K$K|{^*qc0Y%IE6H3EuHT?Q_(lWS>?Ep$VqeyR7kx*s*3^a7!O-k8J}QnZC&s4x8I|tYze;syVzY7Y>U6=L#^@?Cbz6QHlGXLWof=Q4PkSqF~qYAo@5>@cu1b)OCFfj$O+)aH|dzm;(v#U@v<@_fMdniNO_0t1IIMYbIgq<{VS0=JqPKGWe!^MIh+XSw)Qd>T$e?;-+9K$ir?}q~W&u0_NBm*%0gxjdBX^Yp~gd;A-28ahr;deJB7Kf8tDj0!l&Lo|$5eC15Vrk-=0B`1#LzuRQ+X=a|eH&P%X$NA7a^YVyS-eeL#LI@<_lr1cH&=J@f$@614tH8Atc1NHnGY?(*zQZVx07V@3Y`0ScDc=XRWlaC+%6?{-mJ}q5<6=J<I1=tc@>JE~GSw%}1eSlhIkA4KFJdPJeizN%e0*@d56P*+Jiy4M6K(RTW9#Q?*^xxfsn~y*EyP0<kh%mtjYqKLl(XbmcfS-871+=mE_~Cy7BT2omF#d*--h2G;e?k)v|G@4f5D+wbzzM_e)8x^#YoHB`t-lj&f6m>^o$;f8nc}SdRBnh+vmne9#&Y=hgCBxgnmm5^J};y<K%BV!vbX>E;dci$us%WkzLC>n^61_25+hjo@kc+7z&;CCRANl}&%FbH{SE8|zJ2`g?*KO-YCZ5?;pp`_6#9MwW9DMy5&?C_cMRtZ0R7Q>ROiuq^KynN1~<kqLI#;+I1u#1g^HjD^nld@kp;eBCIUd20Ync&R}o|ifh~yCqRy^&>S;{<2i|YRdz9h&0j}hq0Z?4F2i|Mc&I9i&<-WioCf3658*1hr;4XXMZES3K^mk|ZKYppCno@*ujubbz&iA+ez&kxI3H6!BzfJ8$y(@7^y(#qr{mL-Ps8^-l18>$1DpMvEq-1i>!W_V)({%=q-b;kI_`mzRzbjnZa2RTIw?*Q9hn$78qPN}Vs(#0rVb-JeZUW`uy;XRVkAD34;rD0zt2J9Wqwa22S%m7*+dkZD@#^h+za8Ga^~URW?*&uhOFAtCA9t?V^%_JRJf1T`A=tK)!v&>2#K%!6MkQ)qEaa-ZE1b`5dap}z<X4nnw`y%W!mVy7YP1HqY(6eKjZ?aV*f`}PsCt=eckRh~T6gV{#YoAQRqr@?aBX+4c6G4d)bw_M+w^7@(`CIB8z-TE>}}#uFW{=kt}(kRMif!(O?ee9U-yk3b~&DT`(^Em<u{$0OGOqJJT=f!ho=R`2+s~{zRS+8oWbg(mJCD^M(H>|WhZueair_3L<yHgKP!g`1I(NKCO8+<0~iTrBBgv=rI(&^;(}*hE`~F^)M@ZD8Rvk;iz1l>m?Is(mt#YS9nl0R&#^UiN`U0v;4)J1J=ezYsC06OSAus+l(XXB_1n;j)$KVVb(;uEm5!+2dcll!lB<GUv3$k^P&V>frH*G`fUXSabhkEIro^zmQ~HXulF>3ec)5{-q7j8wcH*g!YD(hC`AtMEGDjeevv#mQN%va*C0YJjIuqMFrc+|8@IoFdTjJ$72e}f+g=3~qaUGm-xWr&WbD$l9@PR8w%5s+&##Z(`Jy6w$(gRk#^??pER6J);+V+#Jxjk-lipU)5azC0QC{$c%(qC-1dQF77l~kO0%g4i|1KxWMvVhKw1h0cWa70E`H$0JchFF^b>!eiO2HMdE(4EyUvi{7DLe+ERIFCg&7jqwxa;4r`a(2n#sQq`J!l-GO7f)&A82!_vI6ALGu9Qa&SCTI|MAnQ6=N>4lhDY~Gt+;;D&E|O2YvOukrsrjcxF{rznXi3xYgs6sio&DY2|!3w430Ax&?TBu??$#y@-3%3ZQ(U1$bUiJ>xJZm4VC9*58ZcEDu{+~Qf;xvzjCYu*+Vh&Ct#u9(PzaCYBMJn(;n62YLKh}#t>!dr1PA<Mbe#?zp=sO(bor@dYD`@+N4YRLJ`6N0GvQ68e!n!C3lDF%|YTgJg$v7wG%vMteVAhe2tZ2y0f}oNm(oFRT8}W60{$y+Blxfd;V=qmYU<~Hpt1D7p3^@F9nv)9kAe-qrlQoJi4ur?{02{n8n~p+yDX2SCH=F8&9Q2f@P$H0dT7Zv0Xk(zaeB0GKvl{o=vWnMNLq*1cB(GOivDh^|L`wFrXPl$b9!%EUp|caIbIUQt^o}sRKc1ky$HfL<)UI%+=cE#)&6!a(QJojPkO)%<nMJ7(xgkZIsdQIc><<zYTS~uVI8&n};ZXP+U3(3ZDBr7YaxR8*+-!4duo2p@?T=1CJYCyen`ZWdn3XW7eC|){Hf5#$Be+PlLSm78%ykwwT47Lri5~ue4<@+ClDsXhRvD5E6Dl{~+wVrIoSDN#b&UZc%a<lz}Qm)MlgtT}J_9=P(fKwh-({%<{K1wy7%U<HhQ5@imVc1DOf-30^UDsHhkjC2+(#Vid-r&VG|f6NZ#4C=`(f<7e|?i26FRZ#;hpNLp9s)>KB;R(tLL0$-i{mj'

PROBE_KIND = 'vov_longchau_public_pages'
SAMPLING_METHOD = 'targeted_host_reservoir_unique_urls'
PRIORITY_SHA256 = '1021c7ffbd10629fb3137d0102c8bb43106928d785bca3293125f9a199b55604'
PRIORITY_PAYLOAD = 'c-qZ<%aYqjlHi?Rfl=EUKo!YkrmA|>l&CSaq~4OXT9QdwRlSy#0|QAgiAoY+0iaS@IeFgm9{09pV{@Cs9`-bS*h8=Ff7G9>yN7!q0tkRiNxjyxvPuGor-z4!zvJG$d+#^VdhKm@vvo2bZljCXi{|sVD7<;L*+zL%Wa-44<=Isc=iVxg7I9v9N$S0ef8NDKSq!{vTP9f=t-bg0I!=?_W_0h~z1}j<Hqb^o-{pCnmZRma+~qM;-E6bG^iEefwDmU2)-OKB^Ie(c11~B<>~tNM@j}-x<B#P!nMu8Lw_fwx9>1EG*W0*|r;A^bt^EE81~0!iQMr=eS)rQW!aJkww?>gXmS4qcS0-!yw3}`73<0FyT<f1@veDH_nqZP%sIGST8hRQ5M2lGZgLgk=NjktEi+Gtw7c%ShI-Vm?HMpK-JD4-T2+Bpfb-769Wv}=Bi+Ar|zIp9UJ%5`gS)P>FLm1sG9)5V}_r87c^f?rOuQxpp{`pT~VYj6}@q7f|(W>062hny53m;);M<4OcS0C$dH|s|~PsgL*4elS^r(X~Kf!W~jRg_-r0CTYAUsl6suLr-ZxJsPbHNM+k<bXoMmuUg3p2tH$BdmK2rEYt@9?a28<MJxYFT*&`vpl>^(nT=OqpLN-ApSV-PpG{`Tt>+nW>}PYfUigE>?+QKK7CD=UQ#4!QAX)J#?J$f+Zg5C?Fin+gulh;{rguhus3cS|K%|tZR+_&u~sATKc(SkS(?Ino&t7cyVBA8TonjQRpOI6c?a0m-UH%dqxmSk4rnhk7?)Y@(R1LZ(I$r7nyq)(x|;gH^Dm&_Vv#55GV_*DC0^iL^~)<WkE(o#{k~%ei?pc05Jt7hV6P_y0|1&ch6AJV=j*J1rujPz#k)$%6-)xV+QPhJqed%Wp<bw3s`@sXgr?&1&~{YePg$Y%)v*@F(I<mV?Tq;f2zH!a#354afJySen`Mh@Zd=GS?EFjF5fkqNE%4<4e!x_@%qS`2O%e2OsUnqyYS`A<<am7YwRhjcKj+rQnrGQ%0{sKA+Lly2TL*bOD&i=guYziRGyic=CgnQ*m(_9q$JyCGKRLhu<j3N^YM}5oyQ1`Fu@|S&3{VKZJU;sI82&r~yrLfaR*!kS&Y}hM_=jj+#IgthfFq#kdyl<`$G<xP%wBjif05018$5edZCYV!R1XmUVcB5juQNsOMyijhFyf*0dApA0Mq?(f6Zu{P#T2@Y6jgsM^XrM#b(zO8Oa=>$uqse7I55;ytb!i^&j3J3=skaghem*PM<4M73;d%ata5aOWIgDgrCAvuN)~XcPA0`XPqt;B=2`nvUa#Z+IqcRE_Ph=3PU|p7DDyxN(8niZU0h+g7knSBcQFysftpoJ!+eXUUS21zu0BrlU2N_GLGKYzp1_|khnu8KE}~Kq9$(XYZwjE%JHy8{vyjy4los?c;pLN`5CbD`lOdwCU<j4p#yM=C4XlqxM;Dy6-iJ6(mV8cj)#bjTvV=|)#rb)c<Z%tlEMW9`yXU|d#KkHsfaQRy!7?uAD?9~v0MZaBKq?dkxUT0wN~0V|@hWEeY?96`Y*X)Q1+Tp=lCk^)&gR?G)3+YC4(BMqmiRYp5lVeXvK<l<?sg=@Ft;TTpNis09o}^KcY7W5fpY<tCD32Ff@%gRY6(M81Z(2qDO19ME`<l)Wna-QbvO~&uMq*}abDuzXjiWAZ<)h#hH?7AXMzT+|B(g=j90Y5fx@V;AA2-htT9Set2uaVq;G*}MCS{GWf8A6z>sjP1#CQ91?x7T@{@pQwA(IVQGy!8oMo8h8+iBc-}v9b3J;$mf&g<>Vau)Ef-Doaz>jtr29J~xXxU{5RJw=)oIO^Anx0JsLR#7Q8qOu89=ODU4{|tFG^1eD;5G+FMcFV{QBsD`ChQ|1xE0r^H%|Nkpn#lov{)x;3~~qTOP~?lO#z+4+Q>-`$QUlyi-2mK`M|3C=L6bYVF5cSE!Wpmq*{I2!CYyT6{S@}%{0xfpk+MmN1F_2ah4`?UUVEaeA`4HgA-Ui8I$)_#ZmwP&o-l7njjqt#shEkO<zxIlmhJ!qml?EPGR7U``!?{)XXsT@s&3^_MS}j^i`P(!aQmMUI_36-UH~=IUYUFfPO5ptMq6V%`da%lE{9%D1b(7q9jFP!hDZ6+bu4HO)|)Nb!=PR(2@aF&C*3dUF9I|ECPHz@E)k~6V!@z9B=7;-#jJg)DUt>1AxV!XZgah`bC5iCP=Nj(xjmSkD0Imj~({Vz@p@OCdC$cbr7?%vY6702A=9JTqOC_SHBCtBFpbr5dhI6>6!;ZSP$zNA=jBi7b##&GB4<qM(PbsY=Q6slElDWU22j%i!YM24T<!cPPyDyJQ)TSj!S5tzyT<J4#z2=&QvgMSI&`O3lBRzJI4o@8_b)Y9B_eS7dpntvplEkhnsAH9|&GRn_=Q}I{B70Sb8yl666HxCsY~<;&DTgX?<6>yb3oyc7ihGHcnCKZf_nS7Q(WFG?azNNQ6q=!F@1sWTgcPmINl=#h*jI??9R@J`wRGM4|}$AK3keH3o|22K#g!UBif8gREwKZa`tu0AhnHbu=vqe#Xm4n^dN+%55`PE+DrGa{We&%pwYoJ;T4m5w>*R>`&MN>tuyPU-b{i@FE7*77*{|w(o8;N(vi12wb7G8pOvHQRQz1Y^39UbD#j-`aU#C>!f`Y4mbq9h_s<JfGo{Bi=qM;R}t_yKuu;bj6U{=g8B}4e^3Z~L#R6ultyMrO$y%eFM5YX%>94cYh4oA;^!mK6TyB{CtY(0Oj_Egm9-FXo`O2ag~FXUiu~%X;y;xXY~m)<pbRB+4T9q_i?#g6N$BtLk2cBF+%TWpBA-T6eJTk~Oyx)G77H3m2*#loqKg=KZ{90u40Nto%?;dYQJ{#k1TDN!IMPx~MH6>6`DQ#mA9z{;0ved8D5j{aRjRr$L@iSo9ssLK3pm}U7<AxP0?O%i3j#480&BL#L!rrpG&cB9@LIDbHIg7;OX&C}gl?m8fY*_d>#?MvOyV{UR9JS+On0K@Bu~yNbXv%mkP{Kw__#?C@$=`h+HRoimv>t$^PIV^($!T1BnW5?gX<q!7Lv53fTDjEwkwnr9MW%mVwf0^Q~j&o26VNJNX1d(yE_3T<ha49xA5?HJdUiL<_}26&UneYyRQ^p{Ibz}r4(pjF|@UtlR;?oEGl9&2c#giBp>4E$edcTZY#lAmE|^6C%{w=0)2*Z6H`<ubZS1ksSm7<R-r2G16iNaa)^Ws4`|U0;&0Tc#H1-F*bk#+sUGW^D|pT$IJO;}3V<p^wH|=&Eot}Q#gRU9``(eaymupPX#&f78{P<f7s&;bQVPV|(Zw(K8m&vosLVFNUeB(}xCp2pR82$U(*1xeX(Keq0Za>ObKq@|98SmmJu4%POy$Q2Y<t_RNIsHa0WXVHbn?w_r>=pwv|T4<YQ<?yz8*gucwe9Vj%08^Oz*M&{Mh^Lc-(8-b9S5E6mQbDd;b<x@Bl*eSv!9Q2BFJdJ9TqyEk^th5r1mwWuNhuIOQ$jqh10}-4J-ssjyug8|}03rL*BWInV&I-;0LESXKkS9*jXIcrZS}|2@F}{UiSGq0*^aka#7;z_GTZp#gxvGO^{i;z+rXv#Npfeg^~2vUtUT-}p$0EjGW*;8FU7XJbj>gZkkhKb7b!b$bUPr&hirx$iPj)!~es&q;01K7eObr%5|aVE)GYBdCH%RYD4EI;0f$kBgE3hayH3QobvFi&C^C#pn)FFaw=hp*<!z2;j-1dC8Pf%9*<a`2Z1!6@I8zXp6M5@0v?@k7o-^bg!E=slX5ROfsuZ^(&JxX(55QAXWL^!JPS0q1X+hR%n=G62ok%^f0x;zUv%pV@#-~a3xEh)?YUSqI%#zs%$h>zle*jp}<<$y@TEGlb5-Jrb4r*{gHNm-bHH|AinMo81<(<_lB%r&vkAu*U?2N1Rziy*d=Y!jx(%jj5_oWtb1E0z^w29TJ<&!(b1kQPY6nk<N&D)h{t2(A$#VbtV@$Yuc;83^bX?4MMn`Ttx2f0CccUPq_*$5w%%cL^LIcZx3s}DI;uG?{hOwodpq%N;Ki+P&D?GoPR*oPIn}W>5p~)so|5Vpg;UmLax+)39pVsOqHKOoYpxxf<p_)46Rfv%?&=k5-HR9pwj)jIa<>kJU9FfQ(qX8!0TYUB2TEX6vG3EIqo@t`S^NT=^|hDplnV&kUO~Hb^6?{b2#>VEa~01ow{W_`5daHy?XgN&tI*X@XavGzBP>Sjyn?W%m#T>IUhF7RcZyBnIfWFHivtk-O$0U?m_9)dY}82+E@5b<9ZPM(%3L0Ro{wk9?TBkclmaFu1=_>l{8|D>ho`d!gVooF_v|LhFUgMs$9LrU0=>{4nVQc6ZPMj6sR~u!A+$~`LM|{s(MGKuygxfD6nX#@SuuWI`3`Y8hmpXqmKs{A9&QD$YS<&1u5I;;K9CwwP#Nvx3X@e8i-wR_6IP%T7c9sBFs%uIGG&nf`z`|6wKi~o!0EAgggOa8&NRNF1yR&9%dbZU@#gJ9Ba(@(VS+)R7N>#(un6mH7WnrWZw^Bjb|omf#*xQp@qp#JOn_ZPNTNyS>|C2))uI{8S9Q&zXocOfNn6Q9ug2UFKi^zbT39t8;bP6k>b@v7mQx$EnmB>|Ss1~`q~gG!?Y>YQ^#(DxPiSHdU|`#R?u;ZapC5hq?v=Mm3UEE0uS}e%kBPZ*8iT<BAZam+HTgOR+r;FEIclpLQvx7SdE8gL_Lo}~l+=W>RuoYSE-c+aJ2k3mpi9*T5>9Z2iBK&B<~kup2QLJ)q|g^Mv9Q?+*!QwHuuaSbAyP{Ejtr(8R6{^b&det@IgrCR+Sr}5wLyIvp_64$w=IzA$ed&4toEc~h%kK!Oc56m!vvy|hW+z9fnpUEtD_?MC2qsXn%HKdPD5TME1TtX`1#oL8}F@hO9EaU?4zQJ5z8Jpm}ZZ>3`D;C3MZ<0f^`hUweS`|l-J~h!0T>^dE__=j!4=}7$~2SNnIK>bnsMSK*ZW1%`ZtlP1XyfnpD&+ui3p)4I2#c+Sx4{>D?wrD!|Jk8>&T>1TNthsE4t3Bh}xx87<j3%F*-)ZKxyb&JFO+meI8AYav~#yLutpbJi^^I#FMvPn{IhXn^^zMh|4*d#Hh}vgBUxLLF)_LrNj-v{8$oh7C*^lxnI*N~*SyC8`!tsL5x-x0}A~RYI#}udAJ<^)A(Ob1RPM{|$D-z0axYzzcB4@pkXj%y#T&`(s@?cSvG&27NDcWy@lQ`<yaPh~UKysr?CP%K<#~*Weo*gW+X`{u>T5!Y9Bs+Xm!kho;Pa?{90U_!Q-fc~WV;vtn^<9<Jw@nXkOCLJG~v!zG-7=-=iyrhB+D=l*^1u2P(Lh|1ZiDNps<)YTkZ<aWsF(vtVut%L?<srt)%b%Z-|;aU@Otqsfa%N^}St()M^iik?GY3DUWawL~Gk^A(rB&bGV*+`j8@C`FCXr{4OpjDruq|kxt&C-%6u}>*U00r4=0geK!279v7bUY>H$q^$w>Y&)H3X&Z-6D&vtFfWzGHO<oDGAiL<XT<Z+4H?8gR1X$f(;=!faAml%B#lY|T^Tya0|9ABC4i6}GX;LnwnU2tNE$T^-)hgP1~GmIP68mb7b112=#)wfF^6VJCo(p(EF9gqBWn_lS%QR_#yCaQY?Qq99gT~#25W_7=;pX&SbESdk&Gs;U;_V{T5y1_uE`4*(u%1{XXkeSvM(BLqlccsY*jsBR0!Acb)-i#2nl)Z>yQeY7)<_Hy&gI5c%=P(n`ale#Wm`_g{YNM)Ndg3V$|XWbjd&`*Vmwm-)&%nAYZWMn#7z4<`Mu9HZx=y;tG3l&x9c;M9VM)hwt@rz+R@X(V*aBx5iDn&Mx4@1yVM(1zGgXLlvz4-j_AT!2&S!2aILeCrDL=wO-Cc%8T47HW*(3MKHJ>9PUVu79Wfu({t70v~yZ^lR7>Y`AU4jQ0FIDBsj@&Iq2QkuB?za#$Ctpq(N=Of|Q4sRIAro44fk_(f?0^VMk!!yo5s40xRserUf>Zk0ue2QggA}Yzwe(Dk!01(gd!rK_XxDi!R%huVF3K4B&G_qkK_M-K{aepo;RYHsRfAq4w)Ax0<MbJgg~2Q(W!Pm;Np8qyo;CFe>IrqMX>#X20GQE5j^OEu)QBHkhl$WT$(zSF6+Qz@tbJ9XuJ8g}ixK+WH!eU^V@Ny-&WHddJQ00?#w`<GiI<9sFkv@@=Zqz#eo7u@qpt-_x8rrNx*8&FwiA`c0p1gh^9KF-byW2zT37WE@71dc6|<Ax^E#O{0Qe=EQH%h>$GI52T7cq)p|)tm_eTKafqQof!^l3Qu#hEsN_IPz+&Sm<8|!q$L3AcHm%Zm|-+k$C*1dCwsaDRZyLOSc3*5F==Nn6owt%V52RwnY+*%VM^^?{GS?!CY)}>Yt~N1X^oh->=AwJ7)Oz^z19z+E>hDR=aVMAJYWPWt{Zu)y_Qq3n4(iAx-xw;V88R@k#;F7rVp&Bre23@d2lj@ieG;->PyVi^!OpZf*`aIY)MmO{RZxUr(RpUND-=ZAcd~oH1EQkQWiY96oMA5j)V(to(_dX!&+<RFd#nfL!H0-fP5WkN66?GFxV<kbsN}zP{mEBz$#w7&iWOjN@PK;9fOz@Z6`Pw9Hyz`9Z<}?R(WD_5>R*|29`7xHGahLY6ACb5o+Dz=WGLN0vtRd!&m*#7nzAMfuQD#ZU)JL^q>b_y?2I>^Q`A99DG%lwRK9={E?hve?((v=VYpQDTDt89bd8_QJ6#*_MtN-(u6GS7n*>In0oAuySjZ&g&a`BPSf}zszHu7jwMV|dI7>v{#wUVsr1+iF=`G*%C!q1*c$dJBa%?VN}cux^(Nqm5+oiR-=*n?{6H;MU~g$`AFMw8rr!YURdl$ALx5A#DBU9D9T4Sm6<vg`5FzS{gOJ=J!q9PU`CPiIy1c2H0Zu{VnH5;)=s*lv6HiCx`DVl-vOa%{m_^@)DRg7S(P0XtbeUYVb*1QKn=6H*JIO6FQC<f^nNTOJVss}u;8h~SiYhb=(7-1QL21Gr*yL}oAtMVpj9eEZ#ssJaX);QTROS!zsJ%Tz_br)=6vTM?;yNR41C}|XGW8$O1d?FE)voRJmfP^C4UbphP*0C`)qcm=26|weOF%g)k$c7+Tr&<<pPIUn4yTNRT`O!?2JxDZC#7+}qg#c0ZJueDt}d^NrmLfPep%|_E^2MSR<8yBI-79$b4e_rR9RD7DHfO!(8vmK^}fN?s1aO%1K*ZXQB*0t0<u0i)}C82<Y{-70t>dSX$qtiOJ-GgfEv>oHh%7c8sv@HTI!q^C?)FX^Cd^Lhd+TI6SZY5?{B;}cpT2@#NwHOnIG`w0w#?*X0AOU?`>vTcyWLb!r}v9cNF=$O4!F6J-jJXgkgfKG-PATNLE3UhSp@|VRC47MuFUcmX44KO6Tz9G7jt)wrae%{b(`UhFe*Eo|^e?&QX7|)h#B%kJ<{fs9ac$TxpB6P*ugL?$Qt?0OgX3EHsM)>c}%v{(%i<;7A=vtd)pG{FkisnDNF4rRNlcKwt602y*#>pL0;*>_B0LN;X43HMSJmq<r3<&k3ukPOzSt&M@G<U5+I51wAN$2pBvx`y~G!!sA*(0)FEO04PZzrskkvq=E**aCoFPEqKkq$v_*ZkkD}~i<`LC=qaY3;3yEu=dl4kSjorgB+fPfIog4hyDhCMx>xGmYqw1Mi#AsW&=^jwV49;yvaU5@2lhMTeKq>Bly6z-&e}MO#-VE4M;l(Rjid3PX7o9H&Z+>gpn^3zMt*Wb9^WdjW`|CI6R5HLYzgqa!|916GdWa{udJ8v6wd(e@W22AW~ii6U`U1G0qUV0YX=9`+uiI>Nau-tT3=(zK#bsF?*wi65!(&VN=r;LYdbgjgc`g+F35{YV#L~izrtFJegpd^G1|;5WOGmyDxs!6O6Rj=l&&|UG+B)<vJbwShisbv&>>hIm$*`B_q4c6--fBh#+tQA8gwoLhylEu0~Od&;T?%E_DQzaSKJ)sJ}JR0t6VA7i>?KfT((1nLkEfjGyw6f=#UM}5e;NG)|xY{I@SrnJEt@R;KUP^+O2zBll^r<jXlytvh=!;>>oOiY|`!5`S&HKd{H;>FDtKfP$gO8CsQdhIV>*Io%P9)u1SKL*!`uxeY>M<%}h_2VF_8fdINSsDKo-&-#OKuqpBVhHSQc{Jz3O2xuEWlN~zj(^2_k&cIe*5hTiRXpTg~+Xw^GrYijY1bH%@k2O(s@R4Hz#Ri#G7P=dn~K6U1txnmIFAqFa5_PeBya>rg81e^)CBm7b}swD)q1bhtGBL}d%K;}6R<)AyR!x=I{dh{eEWjMJ#q82nQq9tH6q-q9q_WS)k1X_*~5&11rp<PFjgPZzyAPXck*RW=4PNIoZn6<EPd(hR7Ip&e2uJ~(eKLNES6=zC}r!pF{Wn6?BvSXyfh_aZ+$h~UcJ77$$Z1%PbL(59q4t4HzHT>faS-HmfuTS%2xi>bb=}cx+xQjm-57>@Qx>M_Sa|vvVS^7Ko3RG@)#^68E^G-bDY)I6-2c%aijjJQ5+*}-8odV>gj_I`0ccE3yAEmunRlwmmsFL|Zme7!Qti40I@SfdlE`$y_M!F!4hRn_eS}0~26@+0p?71`Kui&Y2b%i9RRRViR61S$T!b-%6qxmXi-4JkKp}6BxzTx%S?>1N}pnAW5_vVN9FW!ZxFW#Sqe|q!$g(-mIP*8TmQ8*ZVmG2XMl2NM46*~&T5Ha#O<ttQkyk$tD0_kAmf#wUT2bDI~8LA~=c@C9f;~=SF&;DcXvyQbrgHe*3r10eS={Go)F5-_qNzP+)@Onvbfr+HJFqh(PbkRN7Lfku*X^1blUM?-Wrl(kd(VZHWg!W7Wf(?KHbjgL0Z3lklm~9|2j=<-c%IN_z?2c|9fjUPCmD34w3Z;`pZH&rE%U$^e>vGhYCTC$)&B8YfsK%v}flb<VrCbE+lQ$a5Zbg9g&wTy_NND`-%r*bV_?mxV3L<x+`G!!68LBbmvg$$Mhk9GfA&Q7$7TUyM@LX`X?rW7LZ6pu@O;`+S9u>XINwL*ofWa!N&T|3oWM7c8Y7?zcMoaqYi^ed`QnbD=7YfS-DF_?&bu)pqR7t?JSNmt<bF5Vr)cA#;8hETwK~Py<SjdcShq~n1@c8^p_WF5WL@R|*+RR;J^ni~8wR3DAABXb{juTyy-8ixjH4nO4(wX1$jYxL#8P^e71tc}-AhK-`&Jho(-3b-|u_u~1?@tjp)YJ0080vbOq1GWO<zX}f;7(@iD82NBk*YD5+6QvdM(oq*1Ln8Gyo*;+j@iJQ;bgU$R5ip)p%Psk&7HTxL~&4>X2HZy^S}Oky7Csg>tFx!UrX<D_1FIe+R62=|M73v-v9Y;|M!3X`j>y5d+D!#`R@sofiJVa{*MGQM*aF9|GOO7_ps;)7zAexC&|+Ttx*v9mBX$Vs-rU5gLq6YL4Sp2nmAN0F_h#Ho4%oCnrK=(qGwQ;11D&_r6BWvvY$r@-0F~@LP2i=Bwq3-84Y+k#FH~uNfyI_evb@i?P3=riI=*##YR{AwhZ<dsa<K|P*)J2dC&zKG;%Dh?s)L7vmJ#$t#**QM3=BA3;EmOOVo`We=~08u&ON@t6VB;idv<bq4qu+?5j(Y2A-LuzMjI5g0e(1Mt4xi$YW$`)t8>BwzN6VQTHmIG13&fEmj)s5~=<}JbO;xet@C$4#=?Ft^1Qc{jMbNJWp{8b0jFQ>9<D2vXSz}tz44n#iiOlCX@s7p;3j%j7h6pC7`{FK#&cMg+{3c6Jj*nGFs*~>MQ5eVRex%<Bl?wJJmIgI-R%zJ{6p*(Qq2{yXIkGMj-fXVr(BSj(}_If+pqKJ6HE46<JhQH-OnMkqnepe%(fLD^3Z?^JOcu;#;k=OG?m#5$h~$OoOz}7u#^YS%4oGvuS_8qNfnvthl5z#ShVobAG@d$-+8T)Z9!4;+$^fa4_GUC|8|h)meLP@aDZ*1`%$IR)v$FfHo3D4B*Pj<?x}uC+u49<xdT}{(=#{_M(5GRInJMu4k-;ejUEv5^o;eFx|QxZPSZtwmdVe*~gn}@L1hkf=wz=4+R_2$?z0rm$euL;CzINL42<<i~#qz(RSaxcj{sHL>=JC0L?-1@YOwzk30%o0T+ISE>8UWgjn`w+;(>YLOny?Jgl!rHG`Xtx3d2=uY-{G6thbiZt+(~)|#7JXI7fM88X#^+e$XHkNu>Ssa<+*GY959rB!SdM~t+X_Zs#kXVfq&s{|6Z!tH$0nkBYYP^2?gAD@*3QS;z`KN%2>1pVzf3su0sQz`i=y7lU6Be}Y+U9Z(C0JREwZPHF7>|$@yFDqwuz4kW;XAXnR+7i6PhEscU>oc7UynQ2H91~N64+Y&bAa@IMQV$P(CqTwfI@u#|)XU*nu8B2lG*+Cr>L1hnk`76%*)C5PD$raN9r7&VLsjgcylYskYyTSEI)XfdzPm*wiI2#4AjtiH4*qG<a(CvxkRu*SS0FRx0#&$l^N_GjV{l4Zq(N0LZM{O>qJf9a=%srCl83Hpt33-{&-5k8FmBvfRzWE%KLtZgpDmYi8L1!@MQ>G&@aq9($`P?14UWAkNtjA416R`K3YQ!hbYmD=ya(j1gxNRX;e;Oe%$Rt`w>0PG(pM3uV}}i!reLm#%Z}uVngJ#vFO?<fV7zNKGenVkUp0nVvNWNu?owrMJ_~cmdd%vknhCiLJ+&fABSe<h)<>`$;x6qEOwJu@wC|t337>!W=9#*#K}*2gQbqoy)9x0m>(1zKX*YxL<lXKymRgMm<JO-2@!L1A|8sc2G|WoRzWwfL`0U%K|D}D<g4{+~+xF9Ht0DhGxpz&ONo#}bJA1XAU6m>nJ%{itw0vEwqvIt8CBl96Y=-roQ6*rW>4H7DVLgXTBS0LHdx4wDXA#~W{R))oAN5sw9(uPGW;Wo4-zAI)^)2Mf=#FBdNO7RI#HZA+&cZ-<HO?3lF3m5|6oQF!&u>jrvDuzw9b4mgZ59K^Q*qUJ8Dpw4<LU7okY!BoMFnCBLQ<{wo929ct*(XEaFx5{iQ2&HsJt8=kH(`S8;xOx0l$M+!*TiBujzi2>WvxlF12TQ0<Pl2lV&6Q>7>z?`JvuqS`-hUA!!C5fx-{?+cy8&`UubX)oS5lw!x7cODT9h)=hl8oUwM#9ISpjZnXAi6qnKUH?|u6s9#rY|9KIiWzH_A-W@mFT_zvF<N{?M=(6;xn(m+R#?Ty#oaiF->ZH*c*xG@Wwu-7(CkEivi#ovP7#m?$#1}?2daPI4f*UQkUnP0gz#jY4u!87qbx4a7hQ>?Nt3pVFHcX0fQkAAedAef#K5mCw-5li@xTfg&t5*RxQA^;d<R-pSt=Rw%NCdD(8@%7Rn3`oZ3CCP}CQXp<DyBoJtGK1cloAa(dz2C00XPQ>Hl?eCP?BUlf_tKY<@Aw?Vw5OeQPB86Rea}-9`IQXa4Oa2Hv3M|B6gejlTFt3oF92Sjw9Wzw*%=3WCUb>!IcxdS`gvX-RL~Yem#OIh8<$#g@i<r%{1O6i-0V&V@J}4-6Cs12VeTmOW=sW!`H4mAnm#H(dL?dmE=3mD32=L0s_L}VvQk)*pDmKW;J0f0nizywX1;Orfk?~=|)}nLRJC*1-~75&tLrh<<r;UAKtxrefr|{b9C)ZF?=CcQG5XKSI@$yuU<`kIO2P@MjIt;a02FoFo5KqB&`-Y?9Js#EiNn&nZ=0^F*q#XiG1Z5trQUN1W7E6l5bzWeL?R*3NzooKYjk@yVJvOWJLB^^na*>g%z3lqE;gbj}$|2^3G%b@MZz3Qv=?E`W27O^;1!De0Vc^w1|<2z_{=}ZqPe}JU}UG{m5?)mtx?ckQ)i%gJKc_Up7t?(0$xkke~s*#@J&;L7t@%{_NZXTwc*PRAwK(8LS<EB%81((s>afGQw*e6Sx<E*f}|`p`?k9XGHRRrE@~UG5q;w)K)?Qlp4`s_N_@aYDbEbuu3^@#X5S<xbnFhDw?=8x5F6`Kn$@bFK&Mz?Vq=9i@RP&+PaH6cgJ0OX>6wUP4Zyel1dmD8ZtMM4+vNlqntD)&*c5&$KrnQ<NL4r(7J}HjW2CEf~(>c3ai9J6p^8Oww^{5E*cywPanvo0-7(sJ)N-Nmv9IC+<q%zGYr`sOVYKwUT1VZ3szRBhPaB`%E{*#TcOBMze=gg<ZbP+b!;|%q}@h-mcwD14f*X5WkSdrnNohB`_k(9pwbN0r6GrmbxxjA>Y*V$38$nkWjycTtu*~|R&?cx?4VwFenak75L!DpmX@w8t6EbAB>Sy@sPbNU6WGdoPIP#f*1rD*WlSBh`&mudqN?Q4!Y~1?*tcV5@0$_$9NEez>kJHKka0>9QdNEkQ%sT7o`mWU{YBH!Lwq{4zelGph-!O;cOxY8Sn9r4k7{J>X)>)^94Rz>qrNa|bGZ>X(S~x!2e1<{9S$;+aKG}E-(3_^5hDr@7x9N-vDyrG<e=GICyzIvet{~p-354`Tnx9#c3&Ok{V>Fg^3&s!hvUKT#^cJLty;zk$J<)MvO~4xxb5mU38QzQp^G8H4V^zY@%J|k6s!Q!Kg&K=L$eU7*@Su@%Du7d8Ddqk0aLljKtaIm3-lQ9V|bhURzdWsnMd1bmXHD0zsP|112zvKjz$-9Y_%K}fDQrwE}?D}?|<gMe-S==`s~{m;oGOD-)jFAM0+-v;}#uhEhdYC)kP4_7EfRP=>>2Rih2Q)=5r*gLhGquG$rkYp0>!zXp1_tz>M6)`?hf$TjYtu%PWZWBXWQiHc6mp@HEffy?A>1B7FTO{Nd&6=Wl+vZJ@$bqk>XwiodNIgk!e_Jeeu~9mwKL`LDx+>eD@<8s0DBjjW79H3#ASq}E`)gxua(Al4<Tj9)^c1n*EVF;)NHbIICb*{Qm^9VOXLnt^^5Dp4M|Mso@|y2isL_~shl1T1F9G9C?>)gc6nv|~SD$x*dSL1@1er9;zOgePH;`SzPWdGH|Zn`tuCX<2hUh5fuvE>>mzgu5?@rj_R3D_=^{m6N7fa)1<)=m@8dYIBSL(AHK|GgNj~!Fe5n=D=d|RdFA{`q(rF>Gbg<Cur^4GI&g8$0yX#P|y4Ek$U|Yy|bS1<cK4Bd_=E&m&1;_3>GPTtvUZ-woX@}8%05DqRh02K%;IR$-H;bZ!DzVaaCel=J3eTwr*v(J<x08)5p<sq`1aGX{J%9e%<TsfgX=tv$)GZ8(XkrsE#^ceB1}2ih|iA)MQ`?x#wbTl^|I=<N&1<TL?qyIn&!{<aYYDGAS{0%(_`)Go8#Q@yNEr>GT-iI1<#JxYEBi^S$Dg7_hH^Pgz6IMz*G)YT;5fEknb8mOk<nv!JfZD3BjVxtzeJX&DF0bsGL8@r+IpTTB0<6Tw<n1~kL}Bs!##IJW)P03-Qd(a0;Lqf6_ek$qiIYg(c1jCF?nxHDs&+~-)&MW+t9FLFlIa->c7q%yWuK^Bf$Pc>w1@>F}NsJJoS{C^=XcVO8Z)Q}$fccTxj3F2cFS9fs!ZUY&trQeg!@PKFr`L_bB&gk9@helQ|4MdMg%+(VdSEPq7TWfQeG+a~<3lM3=s5_`ZErRI`FtGzK9AFT|;mzfwcFisOYVakuhz6Hc<TLpQq3?*g@t0yk(LJ@*nB{brG-x7<N&;xNCep3Bq@^y_alFNxTLC99IRO??$JlgTs$N^OV#;9A090{nQzUHssk6PmiTmjt)Mj_T|Lsm$9UXNWi5(qf+>2xeR(GcIXs|5qY<x|s^#1_$RE`J'


import base64
import hashlib
import importlib.util
import zlib

def sha256(body):
    return hashlib.sha256(body).hexdigest()

sample_bytes = ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in SAMPLES).encode('utf-8')
crawler_bytes = zlib.decompress(base64.b85decode(CRAWLER_PAYLOAD))
if sha256(crawler_bytes) != CRAWLER_SHA256:
    raise ValueError('Snapshot crawler bị hỏng.')
if (len(SAMPLES) != EXPECTED_URL_COUNT or len({r['domain'] for r in SAMPLES}) != EXPECTED_DOMAIN_COUNT
        or len({r['url'] for r in SAMPLES}) != len(SAMPLES) or len({r['id'] for r in SAMPLES}) != len(SAMPLES)):
    raise ValueError('Danh sách mẫu thiếu hoặc trùng ID/URL.')
if (set(r['domain'] for r in SAMPLES) != set(TARGET_DOMAINS)
        or any(sum(r['domain'] == domain for r in SAMPLES) != SAMPLES_PER_DOMAIN for domain in TARGET_DOMAINS)):
    raise ValueError('Mẫu phải có đúng 20 URL cho mỗi domain VOV và Long Châu.')
manifest_path = RUN_DIR / 'manifest.json'
sample_path = RUN_DIR / 'sample.jsonl'
crawler_path = RUN_DIR / 'crawler_snapshot.py'
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (saved.get('probe_kind') != PROBE_KIND
            or saved.get('sample_sha256') != sha256(sample_bytes)
            or saved.get('source_sha256') != SOURCE_SHA256):
        raise ValueError('Run cũ không khớp mẫu của notebook này.')
    if not sample_path.exists() or sample_path.read_bytes() != sample_bytes:
        raise ValueError('Danh sách mẫu của run cũ đã thay đổi.')
    if not crawler_path.exists() or sha256(crawler_path.read_bytes()) != CRAWLER_SHA256:
        raise ValueError('Snapshot khác run cũ; hãy tạo run mới.')
    if json.loads((RUN_DIR / 'source_profile.json').read_text(encoding='utf-8')) != SOURCE_PROFILE:
        raise ValueError('Source profile của run cũ đã thay đổi.')
else:
    sample_path.write_bytes(sample_bytes)
    crawler_path.write_bytes(crawler_bytes)
    (RUN_DIR / 'source_profile.json').write_text(
        json.dumps(SOURCE_PROFILE, ensure_ascii=False, indent=2), encoding='utf-8')
    manifest_path.write_text(json.dumps({
        'created_at': datetime.now(timezone.utc).isoformat(),
        'probe_kind': PROBE_KIND, 'dataset': 'AIGuruTinix/ViBioMIR',
        'source': SOURCE_URL, 'source_file': SOURCE_FILE, 'source_revision': SOURCE_REVISION,
        'source_sha256': SOURCE_SHA256, 'source_rows': SOURCE_PROFILE['rows'],
        'sampling': SAMPLING_METHOD, 'seed': SAMPLE_SEED, 'selected': len(SAMPLES),
        'target_domains': list(TARGET_DOMAINS), 'samples_per_domain': SAMPLES_PER_DOMAIN,
        'selected_domains': EXPECTED_DOMAIN_COUNT,
        'baseline_archive': BASELINE_ARCHIVE, 'baseline_archive_sha256': BASELINE_ARCHIVE_SHA256,
        'sample_sha256': sha256(sample_bytes),
    }, ensure_ascii=False, indent=2), encoding='utf-8')

spec = importlib.util.spec_from_file_location('domain_probe_crawler', crawler_path)
crawler = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = crawler
spec.loader.exec_module(crawler)
priority_path = RUN_DIR / 'priority_probe_snapshot.py'
priority_bytes = zlib.decompress(base64.b85decode(PRIORITY_PAYLOAD))
if sha256(priority_bytes) != PRIORITY_SHA256:
    raise ValueError('Priority probe snapshot bị hỏng.')
if priority_path.exists() and sha256(priority_path.read_bytes()) != PRIORITY_SHA256:
    raise ValueError('Priority probe khác run cũ; tạo run mới.')
priority_path.write_bytes(priority_bytes)
priority_spec = importlib.util.spec_from_file_location('domain_priority_probe', priority_path)
priority_probe = importlib.util.module_from_spec(priority_spec)
sys.modules[priority_spec.name] = priority_probe
priority_spec.loader.exec_module(priority_probe)
if not isinstance(USER_AGENT, str) or not USER_AGENT.strip() or any(c in USER_AGENT for c in ('\r', '\n')):
    raise ValueError('USER_AGENT phải là chuỗi header hợp lệ.')
crawler.UA = USER_AGENT
crawler.ROBOT_AGENT = USER_AGENT.split('/', 1)[0]
(RUN_DIR / 'baseline_domains.json').write_text(json.dumps(BASELINE_DOMAINS, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Đã nạp {len(SAMPLES)} URL mẫu và crawler {crawler.VERSION}.')


## 4. Chuẩn bị bảng kết quả và phân loại lỗi

Output riêng: `vao_duoc.csv`, `khong_vao_duoc.csv`, `chua_chay.csv`,
`loi_theo_loai.csv`, `can_xem_noi_dung.csv`; bảng tổng là `ket_qua.csv`.
Các CSV dùng UTF-8 BOM để mở được tiếng Việt trong Excel.
`error_detail` giữ nguyên lỗi kỹ thuật; `reason` diễn giải tiếng Việt.
Mở `ket_qua.html` để lọc theo domain hoặc trạng thái.
`uu_tien_fix_domain.csv/html` xếp hostname còn lỗi theo số URL dataset giảm dần.
`tong_hop_domain.csv` gồm cả hostname đã phục hồi; `WEB_ERRORS.md` là báo cáo lỗi chi tiết.


In [ ]:
import csv
import html
import shutil
from collections import Counter
from IPython.display import HTML, FileLink, display

STATUS_REASONS = {
    'robots_unknown': 'Không xác định được chính sách robots.txt; crawler dừng trước khi tải trang.',
    'robots_denied': 'robots.txt trả HTTP 401/403; crawler dừng theo chính sách robots.',
    'robots_disallowed': 'robots.txt không cho phép crawler tải URL này.',
    'http_error': 'Trang đích trả HTTP ngoài khoảng 200–299.',
    'network_error': 'Không tải được trang do lỗi kết nối sau retry.',
    'unsafe_address': 'Địa chỉ đích không phải IP public được crawler cho phép.',
    'invalid_url': 'URL không hợp lệ hoặc dùng cổng ngoài 80/443.',
    'oversized': 'Phản hồi vượt giới hạn kích thước của crawler.',
    'response_deadline': 'Tải phản hồi vượt giới hạn thời gian tổng.',
    'rate_limited_deferred': 'Server yêu cầu chờ Retry-After quá lâu; crawler hoãn URL.',
    'redirect_limit': 'Vượt số lần redirect cho phép.',
    'unsupported_content_type': 'Đã tải phản hồi nhưng định dạng không được hỗ trợ.',
    'unexpected_error': 'Lỗi ngoài dự kiến; xem error_detail.',
}

def classify_error(status, detail, http_status):
    lower = detail.lower()
    if status == 'http_error':
        return f'page_http_{http_status}'
    if status == 'robots_disallowed':
        return 'robots_disallowed'
    if status.startswith('robots_') and 'robots http ' in lower:
        return 'robots_http_' + lower.split('robots http ', 1)[1].split()[0]
    if 'html instead of robots.txt' in lower:
        return 'robots_html_instead_of_txt'
    prefix = 'robots_' if status.startswith('robots_') else ''
    if 'waiting for domain lock' in lower:
        return prefix + 'domain_lock_budget'
    if 'request budget exhausted' in lower or 'retry budget exhausted' in lower:
        return prefix + 'budget_exhausted'
    if 'winerror 10013' in lower or 'network is unreachable' in lower:
        return prefix + 'environment_network_blocked'
    if any(s in lower for s in ('timed out', 'timeout')):
        return prefix + 'timeout'
    if any(s in lower for s in ('sslerror', 'certificate verify failed', 'ssl:')):
        return prefix + 'ssl_error'
    if any(s in lower for s in ('nameresolutionerror', 'getaddrinfo failed', 'name or service not known', 'failed to resolve')):
        return prefix + 'dns_error'
    if any(s in lower for s in ('connection refused', 'failed to establish a new connection', 'connection reset')):
        return prefix + 'connection_error'
    return status

def result_rows():
    rows = []
    for sample in sorted(SAMPLES, key=lambda s: (-SOURCE_PROFILE['host_counts'][s['domain']], s['domain'], s['sample_index'])):
        record_path = RUN_DIR / 'records' / f"{sample['id']}.json"
        record = json.loads(record_path.read_text(encoding='utf-8')) if record_path.exists() else {}
        if record and (record.get('id'), record.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Checkpoint không khớp ID/URL nguồn.')
        status = record.get('status', 'pending')
        page_http = record.get('http_status')
        robots = record.get('robots', {})
        # Robots failure may happen before fetch() attaches its evidence to the record.
        if status.startswith('robots_') and not robots:
            from urllib.parse import urlsplit
            target = record.get('requests', [{}])[-1].get('url', sample['url']) if record.get('requests') else sample['url']
            if record.get('redirects'):
                target = record['redirects'][-1]['to']
            p = urlsplit(target)
            robot_file = RUN_DIR / 'robots' / (sha256(f'{p.scheme}://{p.netloc}'.encode()) + '.json')
            if robot_file.exists():
                robots = json.loads(robot_file.read_text(encoding='utf-8'))
        detail = record.get('error', '')
        browser_file = RUN_DIR / 'browser_records' / f"{sample['id']}.json"
        browser = json.loads(browser_file.read_text(encoding='utf-8')) if browser_file.exists() else {}
        if browser and (browser.get('id'), browser.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Browser checkpoint không khớp ID/URL.')
        requests_access = ('CHUA_CHAY' if status == 'pending' else
                  'VAO_DUOC' if page_http is not None and 200 <= page_http < 300 else 'KHONG_VAO_DUOC')
        if DIRECT_PUBLIC_PROBE:
            access, effective_engine, effective = priority_probe.access_outcome(record, browser)
            if record.get('challenge_detected'):
                requests_access = 'KHONG_VAO_DUOC'
        else:
            access, effective_engine, effective = requests_access, 'requests', record
        flags = record.get('quality_flags', [])
        if status == 'skipped_selenium_first':
            requests_access = 'CHUA_CHAY'
            reason, error_type, stage = 'Bỏ Requests; dùng Selenium trực tiếp theo cấu hình.', '', 'browser_primary'
        elif status == 'pending':
            reason, error_type, stage = 'Chưa chạy URL này.', '', 'pending'
        elif status == 'extracted':
            stage = 'done'
            error_type = ('qc_' + record['quality_status'] if record.get('quality_status') in ('review', 'rejected') else '')
            reason = ('Tải được và nội dung đạt QC tự động.' if record.get('quality_status') == 'accepted'
                      else 'Tải được, nhưng nội dung cần xem lại hoặc bị QC loại: ' + ', '.join(flags))
        else:
            stage = ('robots' if status.startswith('robots_') else 'extraction' if access == 'VAO_DUOC' else 'fetch')
            error_type = record.get('error_kind') or classify_error(status, detail, page_http)
            reason = ('Trang trả HTML xác minh/JavaScript thay vì nội dung bài viết.'
                      if record.get('challenge_detected') else STATUS_REASONS.get(status, status))
        requests_reason = reason
        if effective_engine == 'selenium':
            stage = 'done_browser'
            reason = ('Selenium tải trực tiếp. QC: ' if effective.get('browser_route') == 'selenium_first'
                      else 'Selenium cải thiện tải/trích xuất so với Requests. QC: ') + effective.get('quality_status', 'unknown')
            if access == 'KHONG_VAO_DUOC':
                stage, error_type = 'browser', effective.get('error_kind') or effective.get('status', 'browser_error')
                detail, reason = effective.get('error', ''), 'Selenium không lấy được bài viết.'
        rows.append({
            'domain': sample['domain'], 'dataset_url_count': SOURCE_PROFILE['host_counts'][sample['domain']],
            'sample_id': sample['id'], 'sample_url': sample['url'], 'access_result': access,
            'requests_access_result': requests_access, 'effective_engine': effective_engine,
            'browser_route': effective.get('browser_route', ''),
            'effective_http_status': effective.get('http_status'),
            'effective_quality_status': effective.get('quality_status', ''),
            'effective_char_count': effective.get('char_count'),
            'effective_quality_flags': '; '.join(effective.get('quality_flags', [])),
            'http_status': page_http, 'robots_http_status': robots.get('status'),
            'robots_attempt_count': robots.get('attempt_count'),
            'robots_cache_hit': robots.get('cache_hit'),
            'robots_fetch_id': robots.get('fetch_id', ''),
            'robots_error_kind': robots.get('error_kind', ''),
            'robots_elapsed_seconds': robots.get('elapsed_seconds'),
            'robots_lock_wait_seconds': robots.get('lock_wait_seconds'),
            'crawl_status': status, 'failure_stage': stage, 'error_type': error_type,
            'reason': reason, 'requests_reason': requests_reason,
            'error_detail': detail, 'quality_status': record.get('quality_status', ''),
            'quality_flags': '; '.join(flags), 'char_count': record.get('char_count'),
            'title': record.get('title', ''), 'final_url': record.get('final_url', ''),
            'elapsed_seconds': record.get('elapsed_seconds'),
            'reused_from': record.get('reused_from', ''),
            'request_attempt_count': len(record.get('requests', [])),
            'retryable': record.get('retryable'), 'retry_deferred': record.get('retry_deferred', ''),
            'page_attempted': bool(record.get('requests')),
            'browser_status': browser.get('status', ''),
            'browser_http_status': browser.get('http_status'),
            'browser_quality_status': browser.get('quality_status', ''),
            'browser_char_count': browser.get('char_count'),
            'browser_final_url': browser.get('final_url', ''),
            'browser_error': browser.get('error', ''),
            'browser_attempt_count': browser.get('attempt_count'),
            'sample_role': sample['sample_role'], 'sample_index': sample['sample_index'],
        })
    return rows

def domain_rows(rows):
    result = []
    for domain in sorted(BASELINE_DOMAINS, key=lambda h: (-SOURCE_PROFILE['host_counts'][h], h)):
        selected = [r for r in rows if r['domain'] == domain]
        failed = [r for r in selected if r['access_result'] == 'KHONG_VAO_DUOC']
        content = [r for r in selected if r['access_result'] == 'VAO_DUOC' and r['effective_quality_status'] != 'accepted']
        pending = sum(r['access_result'] == 'CHUA_CHAY' for r in selected)
        errors = Counter(r['error_type'] for r in failed)
        errors.update('qc_' + (r['effective_quality_status'] or 'unknown') for r in content)
        result.append({
            'domain': domain, 'dataset_url_count': SOURCE_PROFILE['host_counts'][domain],
            'baseline_error_type': BASELINE_DOMAINS[domain]['error_type'],
            'selected_urls': len(selected), 'additional_urls': sum(r['sample_role'] == 'additional' for r in selected),
            'missing_additional_urls': BASELINE_DOMAINS[domain]['shortfall'],
            'completed_urls': len(selected) - pending, 'pending_urls': pending,
            'page_not_attempted': sum(not r['page_attempted'] and r['access_result'] != 'CHUA_CHAY' for r in selected),
            'accessible_urls': sum(r['access_result'] == 'VAO_DUOC' for r in selected),
            'access_failures': len(failed), 'content_issues': len(content),
            'total_problem_urls': len(failed) + len(content),
            'additional_problem_urls': sum(r['sample_role'] == 'additional' for r in failed + content),
            'error_types': '; '.join(f'{kind}: {n}' for kind, n in sorted(errors.items())),
            'test_status': 'CHUA_CHAY' if pending == len(selected) else 'CHUA_HOAN_TAT' if pending else 'CON_LOI' if failed or content else 'DA_PHUC_HOI',
        })
    return result

def write_csv(name, rows, fields):
    temporary = RUN_DIR / (name + '.tmp')
    with temporary.open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(RUN_DIR / name)

def save_outputs(show=True):
    rows = result_rows()
    fields = list(rows[0])
    counts = Counter(r['access_result'] for r in rows)
    summary = {'selected': len(rows), 'vao_duoc': counts['VAO_DUOC'],
               'khong_vao_duoc': counts['KHONG_VAO_DUOC'], 'chua_chay': counts['CHUA_CHAY'],
               'crawl_status_counts': dict(Counter(r['crawl_status'] for r in rows)),
               'quality_status_counts': dict(Counter(r['quality_status'] for r in rows if r['quality_status'])),
               'effective_quality_status_counts': dict(Counter(r['effective_quality_status'] for r in rows if r['effective_quality_status'])),
               'created_at': datetime.now(timezone.utc).isoformat()}
    domains = domain_rows(rows)
    priority = [dict(priority_rank=i, **r) for i, r in enumerate((r for r in domains if r['total_problem_urls']), 1)]
    summary.update(hostnames=len(domains), domains_with_errors=len(priority),
                   priority_order='dataset_url_count descending', user_agent=USER_AGENT,
                   page_not_attempted=sum(not r['page_attempted'] and r['access_result'] != 'CHUA_CHAY' for r in rows),
                   additional_urls=sum(r['sample_role'] == 'additional' for r in rows))
    summary.update(request_mode='direct_public_page_probe' if DIRECT_PUBLIC_PROBE else 'robots_enforced',
                   requests_http_2xx=sum(r['requests_access_result'] == 'VAO_DUOC' for r in rows),
                   browser_recovered=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' for r in rows),
                   browser_access_recovered=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' and r['requests_access_result'] != 'VAO_DUOC' and r['browser_route'] != 'selenium_first' for r in rows),
                   browser_content_improved=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' and r['requests_access_result'] == 'VAO_DUOC' for r in rows),
                   browser_primary=sum(r['browser_route'] == 'selenium_first' for r in rows),
                   browser_tested=sum(bool(r['browser_status']) for r in rows),
                   browser_http_2xx=sum(r['browser_http_status'] is not None and 200 <= r['browser_http_status'] < 300 for r in rows))
    write_csv('tong_hop_domain.csv', domains, list(domains[0]))
    write_csv('uu_tien_fix_domain.csv', priority, ['priority_rank', *domains[0]])
    (RUN_DIR / 'uu_tien_fix_domain.json').write_text(json.dumps(priority, ensure_ascii=False, indent=2), encoding='utf-8')
    write_csv('ket_qua.csv', rows, fields)
    write_csv('requests_vs_selenium.csv', rows, fields)
    for value, filename in [('VAO_DUOC', 'vao_duoc.csv'), ('KHONG_VAO_DUOC', 'khong_vao_duoc.csv'), ('CHUA_CHAY', 'chua_chay.csv')]:
        write_csv(filename, [r for r in rows if r['access_result'] == value], fields)
    write_csv('can_xem_noi_dung.csv', [r for r in rows if r['access_result'] == 'VAO_DUOC' and r['effective_quality_status'] != 'accepted'], fields)
    failures = Counter(r['error_type'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC')
    errors = [{'error_type': kind, 'domain_count': len({r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind}),
               'url_count': n,
               'domains': '; '.join(sorted({r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind}))}
              for kind, n in failures.most_common()]
    write_csv('loi_theo_loai.csv', errors, ['error_type', 'domain_count', 'url_count', 'domains'])
    (RUN_DIR / 'ket_qua.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
    (RUN_DIR / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    md = ['# Các website còn lỗi sau lượt test bổ sung', '',
          f"Thời điểm UTC: {summary['created_at']}. Run: `{RUN_DIR.name}`.", '',
          f"Đã chạy {len(rows)-counts['CHUA_CHAY']}/{len(rows)} URL thuộc {len(domains)} hostname. "
          f"Tải được sau Requests/Selenium: {counts['VAO_DUOC']}; chưa tải được: {counts['KHONG_VAO_DUOC']}; chưa chạy: {counts['CHUA_CHAY']}.", '',
          f"Requests tải được: {summary['requests_http_2xx']}; chọn kết quả Selenium: {summary['browser_recovered']} "
          f"(phục hồi truy cập: {summary['browser_access_recovered']}; cải thiện nội dung: {summary['browser_content_improved']}). "
          'Các cột lỗi Requests vẫn giữ nguyên dù Selenium phục hồi thành công.', '',
          f"{summary['page_not_attempted']} URL đã xử lý nhưng chưa request bài viết vì dừng trước bước tải bài. "
          'Các trường hợp này được phân biệt với bài viết thực sự trả HTTP lỗi.', '',
          f'HTTP User-Agent: `{USER_AGENT}`.', '',
          f"Bộ mẫu có {len(rows)} URL thuộc {len(domains)} hostname, gồm {summary['additional_urls']} URL bổ sung. "
          'Gồm lỗi truy cập và lỗi nội dung/QC. Số URL dataset là số dòng của hostname, chưa loại URL trùng.', '',
          'Ưu tiên theo số URL dataset giảm dần; không suy ra toàn bộ URL của domain đều bị lỗi. '
          'Dừng ở robots là chưa request bài viết, không đồng nghĩa website không mở được trong trình duyệt.', '',
          '| Ưu tiên | Hostname | Số URL dataset | Mẫu đã chạy | Lỗi truy cập | Lỗi nội dung | Loại lỗi |',
          '|---:|---|---:|---:|---:|---:|---|']
    for item in priority:
        md.append(f"| {item['priority_rank']} | {item['domain']} | {item['dataset_url_count']:,} | {item['completed_urls']}/{item['selected_urls']} | {item['access_failures']} | {item['content_issues']} | {item['error_types']} |")
    md += ['', '## Domain đã phục hồi trong các mẫu', '']
    md += [f"- {r['domain']}: {r['accessible_urls']}/{r['selected_urls']} URL tải được và đạt QC." for r in domains if r['test_status'] == 'DA_PHUC_HOI'] or ['Chưa xác định domain phục hồi hoàn toàn.']
    md += ['', '## Domain thiếu URL bổ sung', '']
    md += [f"- {r['domain']}: lấy được {r['additional_urls']}/4 URL mới; dataset có {r['dataset_url_count']} dòng." for r in domains if r['missing_additional_urls']]
    md += ['', '## Chi tiết lỗi theo domain', '']
    for item in priority:
        md += [f"### {item['priority_rank']}. {item['domain']}", '',
               f"{item['dataset_url_count']:,} URL nguồn; {item['page_not_attempted']} mẫu đã xử lý nhưng chưa request bài viết. "
               f"Lỗi baseline: `{item['baseline_error_type']}`.", '']
        for row in rows:
            if row['domain'] != item['domain'] or row['access_result'] == 'CHUA_CHAY' or (row['access_result'] == 'VAO_DUOC' and row['effective_quality_status'] == 'accepted'):
                continue
            detail = (row['error_detail'] or row['quality_flags']).replace('\n', ' ').replace('`', "'")
            md += [f"- ID {row['sample_id']} ({row['sample_role']}): [URL](<{row['sample_url']}>) — `{row['error_type']}`. "
                   f"HTTP trang: {row['http_status']}; HTTP robots: {row['robots_http_status']}. {row['reason']} Lỗi gốc: `{detail}`."]
        md.append('')
    (RUN_DIR / 'WEB_ERRORS.md').write_text('\n'.join(md), encoding='utf-8')

    visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'sample_role', 'access_result',
                      'http_status', 'robots_http_status', 'robots_attempt_count', 'robots_cache_hit', 'error_type', 'reason', 'error_detail',
                      'quality_status', 'quality_flags', 'char_count']
    labels = ['Domain', 'Số URL nguồn', 'ID mẫu', 'URL mẫu', 'Mẫu cũ/mới', 'Vào được?', 'HTTP trang',
              'HTTP robots', 'Lần thử robots', 'Cache hit robots', 'Loại lỗi', 'Lý do', 'Lỗi gốc', 'QC', 'Cờ QC', 'Số ký tự']
    if DIRECT_PUBLIC_PROBE:
        visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'http_status',
                          'error_type', 'error_detail', 'quality_status', 'char_count',
                          'browser_status', 'browser_http_status', 'browser_quality_status', 'browser_char_count', 'browser_error',
                          'access_result', 'effective_engine']
        labels = ['Domain', 'Số URL nguồn', 'ID', 'URL', 'HTTP Requests', 'Lỗi Requests', 'Chi tiết',
                  'QC Requests', 'Ký tự Requests', 'Selenium', 'HTTP Selenium', 'QC Selenium', 'Ký tự Selenium', 'Lỗi Selenium',
                  'Kết quả cuối', 'Engine tải được']
    def esc(value):
        return html.escape(str(value if value is not None else ''), quote=True)
    body = []
    for row in rows:
        cells = []
        for field in visible_fields:
            value = row[field]
            text = (f'<a href="{esc(value)}" target="_blank" rel="noopener noreferrer">{esc(value)}</a>'
                    if field == 'sample_url' else esc(value))
            cells.append('<td>' + text + '</td>')
        body.append(f'<tr data-access="{row["access_result"]}">' + ''.join(cells) + '</tr>')
    style = """<style>body{font:14px/1.5 system-ui,sans-serif;margin:20px;color:#172b4d}
    .table-wrap{overflow:auto;max-height:70vh;border:1px solid #ccd4df}table{border-collapse:collapse;width:100%}
    th,td{padding:9px;border-bottom:1px solid #dce2e9;text-align:left;vertical-align:top}th{background:#e8eef6;position:sticky;top:0}
    td{min-width:95px;max-width:330px;overflow-wrap:anywhere}td:nth-child(4),td:nth-child(9),td:nth-child(10){min-width:260px}
    input,select{padding:8px;margin:0 10px 14px 0}a{color:#1654a0}</style>"""
    table = '<div class="table-wrap"><table><thead><tr>' + ''.join('<th>' + esc(label) + '</th>' for label in labels)
    table += '</tr></thead><tbody>' + ''.join(body) + '</tbody></table></div>'
    heading = f'<h1>Kết quả thử crawl {len(rows)} URL / {len(domains)} hostname</h1><p>Vào được: {counts["VAO_DUOC"]} · Không vào được: {counts["KHONG_VAO_DUOC"]} · Chưa chạy: {counts["CHUA_CHAY"]}</p>'
    heading += '<p>Vào được tính cả Requests và Selenium phục hồi; trang challenge được loại. Các cột Requests vẫn giữ lỗi gốc, QC là kết quả riêng. Kết quả chỉ áp dụng cho các URL mẫu.</p>'
    controls = '<input id="search" placeholder="Tìm domain, URL hoặc lỗi" aria-label="Tìm kiếm"><select id="access" aria-label="Kết quả"><option value="">Tất cả</option><option>VAO_DUOC</option><option>KHONG_VAO_DUOC</option><option>CHUA_CHAY</option></select><span id="count"></span>'
    priority_fields = ['priority_rank', 'domain', 'dataset_url_count', 'selected_urls', 'completed_urls', 'pending_urls', 'access_failures', 'content_issues', 'error_types']
    priority_labels = ['Ưu tiên', 'Domain', 'Số URL dataset', 'Mẫu', 'Đã chạy', 'Chưa chạy', 'Lỗi truy cập', 'Lỗi nội dung', 'Loại lỗi']
    priority_heading = '<h2>Domain còn lỗi — số URL dataset giảm dần</h2><p>Gồm lỗi truy cập và lỗi nội dung/QC. Số URL nguồn không phải số URL đã xác minh lỗi.</p>'
    priority_table = '<div class="table-wrap"><table id="priority"><thead><tr>' + ''.join('<th>'+esc(label)+'</th>' for label in priority_labels)
    priority_table += '</tr></thead><tbody>' + ''.join('<tr>' + ''.join('<td>'+esc(r[f])+'</td>' for f in priority_fields) + '</tr>' for r in priority) + '</tbody></table></div>'
    (RUN_DIR / 'uu_tien_fix_domain.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><title>Ưu tiên domain lỗi</title>'+style+priority_heading+priority_table+'</html>', encoding='utf-8')
    javascript = """<script>const rs=[...document.querySelectorAll('tbody tr')],s=document.getElementById('search'),a=document.getElementById('access');
    function filter(){let n=0;for(const r of rs){const ok=r.textContent.toLowerCase().includes(s.value.toLowerCase())&&(!a.value||r.dataset.access===a.value);r.hidden=!ok;n+=ok?1:0}document.getElementById('count').textContent=n+' hostname'}
    s.addEventListener('input',filter);a.addEventListener('change',filter);filter();</script>"""
    javascript = javascript.replace("document.querySelectorAll('tbody tr')", "document.querySelectorAll('tr[data-access]')").replace("n+' hostname'", "n+' URL'")
    (RUN_DIR / 'ket_qua.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Kết quả crawl domain</title>' + style + heading + priority_heading + priority_table + '<h2>Từng URL mẫu</h2>' + controls + table + javascript + '</html>', encoding='utf-8')
    # Archive outside the run directory to avoid including the archive itself.
    zip_path = Path(shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name))
    if show:
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        display(HTML(style + priority_heading + priority_table))
        display(HTML(style + heading + table))
        print('File kết quả:', RUN_DIR)
        print('ZIP:', zip_path)
    return rows, summary, zip_path

# Create reports before networking, so partial runs have explicit CHUA_CHAY rows.
rows, summary, zip_path = save_outputs(show=False)
print('Đã chuẩn bị báo cáo và phân loại lỗi.')


## 5. Tải trang và đối chiếu Selenium

Chạy Requests trước, từng URL tuần tự trong một domain; các domain chạy song song.
Mỗi URL có timeout connect/read và ngân sách riêng, không tiêu hao ngân sách vì các mẫu cùng domain chờ nhau.
Ngân sách được kiểm tra giữa các bước; DNS và thử nhiều IP có thể làm thời gian thực vượt mức đặt.
Theo `BROWSER_TEST_MODE`, Selenium thử lại URL Requests lỗi hoặc HTTP 200 nhưng text rỗng/QC lỗi;
có thể đặt `off` để chỉ chạy Requests. Nếu bật Selenium, Chrome được kiểm tra trước lượt Requests.
Trên Linux, notebook tự cài thư viện Chrome còn thiếu bằng apt (gồm ATK); Selenium Manager tìm/tải
Chrome và driver. Preflight thất bại sẽ dừng ô crawl, lưu `browser_setup_error.json`,
`browser_runtime.json`, `browser_preflight.json` và log Chrome để chẩn đoán.
Requests và Selenium có kết quả riêng; DOM trống/challenge không được ghi thành tải được bài.
Requests retry tối đa 3 lần; lỗi mạng được đóng pool kết nối cũ trước lần thử tiếp theo.
Selenium dùng User-Agent thực và retry timeout tối đa 2 lần; 403/challenge không được retry liên tục.
`access_result` tính cả trường hợp Selenium phục hồi, còn các cột lỗi Requests vẫn giữ nguyên.
Kết quả cuối chọn engine có QC tốt hơn; kết quả tương đương giữ Requests.
Parser 120ask lấy câu hỏi/câu trả lời; cnkang lấy thân bài, bỏ menu/bài liên quan.
`summary.json` tách số URL phục hồi truy cập và số URL cải thiện nội dung nhờ Selenium.
Để thử lại riêng nhóm lỗi, đặt `RECOVERY_RUN_DIR` là đường dẫn run cũ và giữ `RESUME_RUN_DIR = None`.
Recovery tạo run mới, kiểm tra hash raw 2xx trước khi reuse, để các checkpoint lỗi được tải lại.
Để chạy lại toàn bộ bằng cấu hình mới, giữ `RESUME_RUN_DIR = None` và Run All.

Mặc định `failed_only` tự chuyển mọi lỗi Requests/trích xuất, text rỗng và QC `review`/`rejected`
sang Selenium, kể cả PDF cần kiểm tra lại. `off` chủ động tắt đối chiếu Selenium;
robots bị chặn và URL không hợp lệ/địa chỉ không được phép vẫn được giữ nguyên.

Tiến độ và cảnh báo QC (review, domain_parser_fallback, …) được tổng hợp mỗi LOG_EVERY URL.
Lỗi HTTP/mạng, trang challenge và nội dung rỗng vẫn in ngay kèm URL và nguyên nhân.


In [ ]:
import argparse
import os
import socket
import time

# DNS check is diagnostic only; failure here does not stop the crawl.
network_path = RUN_DIR / 'network_check.json'
if not network_path.exists():
    network = {
        'checked_at': datetime.now(timezone.utc).isoformat(),
        'environment': 'kaggle' if Path('/kaggle/working').is_dir() else 'colab' if Path('/content').is_dir() else 'local',
        'python': sys.version, 'platform': sys.platform,
        'proxy_environment_present': {key: bool(os.environ.get(key)) for key in
                                      ('HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY')},
        'dns': [],
    }
    for domain in NETWORK_CHECK_DOMAINS:
        start = time.monotonic()
        entry = {'domain': domain}
        try:
            entry['addresses'] = sorted({r[4][0] for r in socket.getaddrinfo(domain, 443)})
        except OSError as exc:
            entry.update(exception_class=type(exc).__name__, error=str(exc))
        entry['elapsed_seconds'] = round(time.monotonic() - start, 3)
        network['dns'].append(entry)
    network_path.write_text(json.dumps(network, ensure_ascii=False, indent=2), encoding='utf-8')
print('Network check:', network_path)

try:
    if DIRECT_PUBLIC_PROBE:
        if BROWSER_TEST_MODE != 'off':
            priority_probe.browser_preflight(crawler, RUN_DIR, user_agent=BROWSER_USER_AGENT,
                                             chrome_binary=CHROME_BINARY, load_timeout=BROWSER_LOAD_TIMEOUT)
        if RECOVERY_RUN_DIR:
            recovery = priority_probe.seed_successful_requests(crawler, RUN_DIR, SAMPLES, RECOVERY_RUN_DIR)
            print('Recovery:', len(recovery['reused_ids']), 'URL reuse raw;', len(recovery['retry_ids']), 'URL thử lại.')
        primary_enabled = PUBMED_SELENIUM_FIRST and BROWSER_TEST_MODE != 'off'
        primary, primary_new = priority_probe.prepare_pubmed_selenium(
            crawler, RUN_DIR, SAMPLES, enabled=primary_enabled, max_new=MAX_NEW)
        if primary:
            priority_probe.crawl_browser(
                crawler, RUN_DIR, primary, user_agent=BROWSER_USER_AGENT, chrome_binary=CHROME_BINARY,
                delay=DELAY, load_timeout=BROWSER_LOAD_TIMEOUT, attempts=BROWSER_ATTEMPTS, log_every=LOG_EVERY)
            if any(not (RUN_DIR / 'browser_records' / f"{s['id']}.json").exists() for s in primary):
                raise RuntimeError('PubMed Selenium chưa hoàn tất; xem browser_setup_error.json và resume.')
        request_samples = [s for s in SAMPLES if not primary_enabled
                           or crawler.host(s['url']) != 'pubmed.ncbi.nlm.nih.gov']
        request_limit = None if MAX_NEW is None else MAX_NEW - primary_new
        if request_limit != 0:
            priority_probe.crawl_requests(
                crawler, RUN_DIR, request_samples, workers=WORKERS, delay=DELAY,
                attempts=PAGE_ATTEMPTS, user_agent=USER_AGENT,
                timeout=(PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT), budget=PAGE_REQUEST_BUDGET, max_new=request_limit, log_every=LOG_EVERY)
        records = [crawler.load_json(RUN_DIR / 'records' / f"{s['id']}.json") for s in request_samples
                   if (RUN_DIR / 'records' / f"{s['id']}.json").exists()]
        browser_samples = priority_probe.select_browser_samples(request_samples, records, BROWSER_TEST_MODE)
        if browser_samples:
            priority_probe.crawl_browser(crawler, RUN_DIR, browser_samples,
                                        user_agent=BROWSER_USER_AGENT, chrome_binary=CHROME_BINARY, delay=DELAY,
                                        load_timeout=BROWSER_LOAD_TIMEOUT, attempts=BROWSER_ATTEMPTS, log_every=LOG_EVERY)
        crawler.report(argparse.Namespace(out=str(RUN_DIR)))
    else:
        crawler.locked_crawl(argparse.Namespace(
            out=str(RUN_DIR), workers=WORKERS, delay=DELAY, attempts=ATTEMPTS, max_new=MAX_NEW,
            log_every=LOG_EVERY,
            robots_attempts=ROBOTS_ATTEMPTS, robots_connect_timeout=ROBOTS_CONNECT_TIMEOUT,
            robots_read_timeout=ROBOTS_READ_TIMEOUT, robots_retry_budget=ROBOTS_RETRY_BUDGET,
            robots_failure_ttl=ROBOTS_FAILURE_TTL,
        ))
finally:
    rows, summary, zip_path = save_outputs()


## 6. Xem lỗi riêng và tải output

`khong_vao_duoc.csv` chứa domain, URL, loại lỗi, lý do tiếng Việt và lỗi gốc.
`loi_theo_loai.csv` gom số hostname và số URL theo từng lỗi (HTTP 403, timeout,
robots trả HTML, robots cấm crawl, DNS, SSL…). File JSON trong `records/`
giữ trace request/redirect; `robots/` giữ bằng chứng robots.txt.
`network_check.json` ghi DNS của mọi hostname trong bộ mẫu để đối chiếu Kaggle/local;
`priority_probe_config.json` ghi User-Agent/header, timeout và chế độ tải trang.
`requests_vs_selenium.csv` và `browser_records/` giữ kết quả trình duyệt riêng;
`browser_setup_error.json` được ghi nếu Chrome/driver không khởi động được.
`browser_runtime.json`, `browser_preflight.json`, `chrome_dependencies.log` và `chromedriver.log`
ghi kiểm tra môi trường/cài thư viện/khởi động Chrome (log cài apt chỉ có khi cần cài thêm).
ZIP gồm kết quả, raw, ảnh chụp và snapshot để kiểm tra riêng.

Kaggle: tải ZIP trong **Output**. Colab: ô dưới hiện nút tải ZIP.
Jupyter local: dùng các liên kết file hoặc mở đường dẫn output được in ra.


In [ ]:
# Có thể chạy lại ô này để xuất báo cáo từ checkpoint mà không crawl thêm.
rows, summary, zip_path = save_outputs(show=False)
failed_rows = [r for r in rows if r['access_result'] == 'KHONG_VAO_DUOC']
print(f'Không vào được: {len(failed_rows)}/{len(rows)} URL, thuộc {len({r["domain"] for r in failed_rows})} hostname')
for row in failed_rows:
    print(f"{row['domain']} | {row['error_type']} | {row['error_detail']}")
print('\nOutput:', RUN_DIR)
for name in ['WEB_ERRORS.md', 'uu_tien_fix_domain.html', 'uu_tien_fix_domain.csv', 'tong_hop_domain.csv',
             'requests_vs_selenium.csv',
             'ket_qua.html', 'ket_qua.csv', 'vao_duoc.csv', 'khong_vao_duoc.csv',
             'loi_theo_loai.csv', 'can_xem_noi_dung.csv', 'chua_chay.csv']:
    display(FileLink(str(RUN_DIR / name)))
display(FileLink(str(zip_path)))

if 'google.colab' in sys.modules:
    import ipywidgets as widgets
    from google.colab import files
    button = widgets.Button(description='Tải ZIP kết quả')
    button.on_click(lambda _: files.download(str(zip_path)))
    display(button)
